# CS781 Capstone Phase 5: Darsh Kedia (250311)

In [ ]:
import glob, os, torch
WORK = "/kaggle/working"
OUT = f"{WORK}/p5r2"
CACHE = f"{WORK}/p5_cache_all.pt"
DOCS_PER_EPOCH = 14000
QUICK = False
MAX_HOURS = 3.8

CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/model_best.pt", recursive=True)):
    arch = torch.load(p, map_location="cpu", weights_only=False)["model_config"]["arch"]
    CKPT.setdefault(arch, p)
toks = sorted(glob.glob("/kaggle/input/**/tokenizer.json", recursive=True))
TOKENIZER = toks[0] if toks else None
print("checkpoints:", CKPT)
print("tokenizer:", TOKENIZER)
assert "unilm" in CKPT and TOKENIZER, "attach the Phase 1 unilm output (model_best.pt + data/tokenizer.json)"
missing = [a for a in ("gpt", "unilm", "encdec", "retnet") if a not in CKPT]
print("missing archs (the course wants all four in this phase):", missing or "none")

In [ ]:
!pip -q install tokenizers huggingface_hub

In [ ]:
%%writefile cs781_p1.py
#!/usr/bin/env python3
"""CS781 Capstone, Phase 1: legal language model pretraining from scratch.

References:
  [1] Vaswani et al. 2017, Attention Is All You Need.
  [2] Radford et al. 2019, GPT-2 (init scheme, pre-norm decoder).
  [3] Su et al. 2021, RoFormer: rotary position embeddings (RoPE).
  [4] Zhang & Sennrich 2019, RMSNorm.   [5] Shazeer 2020, GLU variants (SwiGLU).
  [6] Raffel et al. 2020, T5 (span corruption, random_spans_noise_mask).
  [7] Dong et al. 2019, UniLM.   [8] Devlin et al. 2019, BERT (MLM 80/10/10, MLM head).
  [9] Sun et al. 2023, Retentive Network (multi-scale decay, parallel/recurrent forms).
  [10] Karpathy, nanoGPT (memmap token bins, training-loop structure).
  [11] Sennrich et al. 2016 BPE; HF `tokenizers` library for byte-level BPE training.
"""
import argparse
import dataclasses
import io
import json
import math
import os
import random
import time
import zipfile
from dataclasses import dataclass, field, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

N_SENTINELS = 64
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>"] + [
    f"<extra_id_{i}>" for i in range(N_SENTINELS)]
PAD, UNK, BOS, EOS, MASK, SEP = 0, 1, 2, 3, 4, 5
SENTINEL0 = 6
N_SPECIAL = len(SPECIAL_TOKENS)


@dataclass
class ModelConfig:
    arch: str = "gpt"
    vocab_size: int = 32000
    d_model: int = 768
    n_layer: int = 12
    n_head: int = 12
    d_ff: int = 2048
    max_len: int = 1024
    dropout: float = 0.0
    rope_base: float = 10000.0
    n_enc_layer: int = 6
    n_dec_layer: int = 6
    enc_len: int = 768
    grad_ckpt: bool = False

    @property
    def dec_len(self):
        return self.max_len - self.enc_len


PRESETS = {
    "gpt": dict(arch="gpt", d_model=768, n_layer=12, n_head=12, d_ff=2048),
    "encdec": dict(arch="encdec", d_model=768, n_enc_layer=6, n_dec_layer=6, n_head=12, d_ff=2048),
    "unilm": dict(arch="unilm", d_model=768, n_layer=12, n_head=12, d_ff=2048),
    "retnet": dict(arch="retnet", d_model=768, n_layer=12, n_head=6, d_ff=2048),
}
SMOKE_MODEL = dict(d_model=64, n_layer=2, n_enc_layer=2, n_dec_layer=2, n_head=4, d_ff=128)

PHASE1_SOURCES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}


@dataclass
class RunConfig:
    run_name: str = "gpt"
    arch: str = "gpt"
    seed: int = 42
    smoke: bool = False
    out_dir: str = "/kaggle/working/cs781_phase1"
    data_dir: str = "/kaggle/working/cs781_phase1/data"
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    local_data_dir: str = ""
    sources: dict = field(default_factory=lambda: dict(PHASE1_SOURCES))
    test_file: str = "Phase_1_Pretraining/Test/test.json"
    mix: dict = field(default_factory=lambda: {
        "supreme_court": 0.35, "high_courts": 0.35, "tribunals": 0.15, "acts": 0.15})
    prepare_train_tokens: int = 420_000_000
    val_tokens: int = 4_000_000
    val_every_doc: int = 100
    n_pseudo_test: int = 200
    tokenizer_vocab: int = 32000
    tokenizer_sample_chars: int = 250_000_000
    model: dict = field(default_factory=dict)
    train_tokens: int = 200_000_000
    seq_len: int = 1024
    micro_batch: int = 8
    batch_tokens: int = 131072
    lr: float = 6e-4
    min_lr_frac: float = 0.1
    warmup_steps: int = 100
    extend_peak_frac: float = 0.5
    weight_decay: float = 0.1
    beta2: float = 0.95
    grad_clip: float = 1.0
    log_every: int = 10
    eval_every: int = 200
    eval_batches: int = 20
    save_every: int = 100
    max_hours: float = 11.0
    resume_from: str = ""
    compile: bool = False
    hf_push_repo: str = ""
    gen_checkpoint: str = "best"
    gen_max_new_tokens: int = 224
    gen_target_words: int = 128
    gen_batch: int = 32
    temperature: float = 0.7
    top_k: int = 50
    top_p: float = 0.9
    repetition_penalty: float = 1.1
    no_repeat_ngram: int = 4
    max_test_examples: int = 0

    def apply_smoke(self):
        self.model = {**SMOKE_MODEL, **self.model}
        self.seq_len = 128
        self.prepare_train_tokens = min(self.prepare_train_tokens, 2_000_000)
        self.val_tokens = min(self.val_tokens, 100_000)
        self.val_every_doc = min(self.val_every_doc, 10)
        self.n_pseudo_test = min(self.n_pseudo_test, 8)
        self.tokenizer_vocab = min(self.tokenizer_vocab, 2000)
        self.tokenizer_sample_chars = min(self.tokenizer_sample_chars, 5_000_000)
        self.train_tokens = min(self.train_tokens, 120_000)
        self.micro_batch = 4
        self.batch_tokens = 2048
        self.warmup_steps = 5
        self.lr = 2e-3
        self.log_every, self.eval_every, self.eval_batches, self.save_every = 5, 20, 4, 20
        self.gen_max_new_tokens, self.gen_target_words, self.gen_batch = 48, 32, 8
        self.max_test_examples = self.max_test_examples or 16
        return self

    def model_config(self, vocab_size):
        kw = dict(PRESETS[self.arch])
        kw.update(self.model)
        kw["max_len"] = self.seq_len
        kw.setdefault("enc_len", self.seq_len * 3 // 4)
        kw["vocab_size"] = vocab_size
        return ModelConfig(**kw)

    @property
    def run_dir(self):
        return os.path.join(self.out_dir, self.run_name)


def load_run_config(path_or_dict):
    d = path_or_dict
    if isinstance(d, str):
        with open(d) as f:
            d = json.load(f)
    known = {f.name for f in dataclasses.fields(RunConfig)}
    cfg = RunConfig(**{k: v for k, v in d.items() if k in known})
    if cfg.smoke:
        cfg.apply_smoke()
    return cfg


def log0(*a):
    if int(os.environ.get("RANK", "0")) == 0:
        print(*a, flush=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def record_text(rec):
    if isinstance(rec, str):
        return rec
    if isinstance(rec, dict):
        t = rec.get("text")
        if isinstance(t, str):
            return t
        return "\n".join(v for v in rec.values() if isinstance(v, str))
    return ""


def iter_json_records(f):
    import ijson
    head = f.read(1 << 16)
    f.seek(0)
    stripped = head.lstrip()
    first = stripped[:1]
    if first == b"[":
        for i, rec in enumerate(ijson.items(f, "item", use_float=True)):
            yield i, rec
        return
    if first == b"{":
        nl = stripped.find(b"\n")
        line = stripped[:nl] if nl >= 0 else stripped
        try:
            json.loads(line)
            is_jsonl = nl >= 0 and len(stripped[nl:].strip()) > 0
        except ValueError:
            is_jsonl = False
        if not is_jsonl:
            for k, rec in ijson.kvitems(f, "", use_float=True):
                yield k, rec
            return
    for i, line in enumerate(io.TextIOWrapper(f, encoding="utf-8")):
        line = line.strip()
        if line:
            yield i, json.loads(line)


def open_source(cfg, rel_path):
    if cfg.local_data_dir:
        return open(os.path.join(cfg.local_data_dir, rel_path), "rb")
    from huggingface_hub import HfFileSystem
    fs = HfFileSystem(token=os.environ.get("HF_TOKEN"))
    return fs.open(f"datasets/{cfg.hf_repo}/{rel_path}", "rb", block_size=8 << 20)


def iter_source_docs(cfg, rel_path):
    with open_source(cfg, rel_path) as f:
        for key, rec in iter_json_records(f):
            text = record_text(rec)
            if text and text.strip():
                yield key, text


def load_json_file(cfg, rel_path):
    with open_source(cfg, rel_path) as f:
        return json.load(f)


def train_tokenizer(texts, vocab_size, out_path):
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.post_processor = processors.ByteLevel(trim_offsets=True)
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    tok.train_from_iterator(texts, trainer=trainer)
    for i, s in enumerate(SPECIAL_TOKENS):
        assert tok.token_to_id(s) == i, (s, tok.token_to_id(s))
    tok.save(out_path)
    return tok


def load_tokenizer(path):
    from tokenizers import Tokenizer
    return Tokenizer.from_file(path)


def prepare(cfg):
    os.makedirs(cfg.data_dir, exist_ok=True)
    tok_path = os.path.join(cfg.data_dir, "tokenizer.json")
    t0 = time.time()
    total_mix = sum(cfg.mix.values())
    mix = {k: v / total_mix for k, v in cfg.mix.items()}

    if not os.path.exists(tok_path):
        def sample_texts():
            for src, frac in mix.items():
                budget, seen = int(cfg.tokenizer_sample_chars * frac), 0
                for idx, (_, text) in enumerate(iter_source_docs(cfg, cfg.sources[src])):
                    if idx % cfg.val_every_doc == 0:
                        continue
                    yield text
                    seen += len(text)
                    if seen >= budget:
                        break
                log0(f"[tokenizer] {src}: {seen/1e6:.1f}M chars sampled")
        log0(f"[tokenizer] training byte-level BPE, vocab {cfg.tokenizer_vocab}")
        train_tokenizer(sample_texts(), cfg.tokenizer_vocab, tok_path)
        log0(f"[tokenizer] saved {tok_path} ({time.time()-t0:.0f}s)")
    tok = load_tokenizer(tok_path)
    assert tok.get_vocab_size() < 65535

    meta = {"vocab_size": tok.get_vocab_size(), "sources": {}, "mix": mix}
    pseudo_prompts, pseudo_refs = {}, {}
    val_f = open(os.path.join(cfg.data_dir, "val.bin"), "wb")
    for src, frac in mix.items():
        quota = int(cfg.prepare_train_tokens * frac)
        vquota = int(cfg.val_tokens * frac)
        n_train = n_val = n_docs = n_pseudo = 0
        chars = 0
        pseudo_cap = max(1, round(cfg.n_pseudo_test * frac))
        train_f = open(os.path.join(cfg.data_dir, f"train_{src}.bin"), "wb")
        batch, batch_is_val = [], []

        def flush():
            nonlocal n_train, n_val, chars
            encs = tok.encode_batch(batch)
            for text, enc, is_val in zip(batch, encs, batch_is_val):
                ids = enc.ids + [EOS]
                arr = np.asarray(ids, dtype=np.uint16)
                if is_val:
                    if n_val < vquota:
                        arr = arr[:vquota - n_val]
                        arr.tofile(val_f)
                        n_val += len(arr)
                elif n_train < quota:
                    frac_kept = min(1.0, (quota - n_train) / len(arr))
                    arr = arr[:quota - n_train]
                    arr.tofile(train_f)
                    n_train += len(arr)
                    chars += int(len(text) * frac_kept)
            batch.clear()
            batch_is_val.clear()

        for idx, (key, text) in enumerate(iter_source_docs(cfg, cfg.sources[src])):
            is_val = idx % cfg.val_every_doc == 0
            if is_val:
                words = text.split()
                if len(words) >= 640 and n_pseudo < pseudo_cap:
                    n_pseudo += 1
                    pid = f"{src}_{key}"
                    pseudo_prompts[pid] = {"text": " ".join(words[:512])}
                    pseudo_refs[pid] = {"text": " ".join(words[512:640])}
            batch.append(text)
            batch_is_val.append(is_val)
            n_docs += 1
            if len(batch) >= 256:
                flush()
            if n_train >= quota and n_val >= vquota:
                break
        if batch:
            flush()
        train_f.close()
        meta["sources"][src] = {"train_tokens": n_train, "val_tokens": n_val, "docs_read": n_docs,
                                "chars_per_token": round(chars / max(n_train, 1), 3),
                                "quota": quota, "short": n_train < quota}
        log0(f"[prepare] {src}: {n_train/1e6:.1f}M train tokens, {n_val/1e6:.2f}M val, "
             f"{n_docs} docs, {chars/max(n_train,1):.2f} chars/token ({time.time()-t0:.0f}s)")
    val_f.close()
    with open(os.path.join(cfg.data_dir, "pseudo_test.json"), "w") as f:
        json.dump(pseudo_prompts, f, ensure_ascii=False)
    with open(os.path.join(cfg.data_dir, "pseudo_truth.json"), "w") as f:
        json.dump(pseudo_refs, f, ensure_ascii=False)
    meta["pseudo_test"] = len(pseudo_prompts)
    with open(os.path.join(cfg.data_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    log0(json.dumps(meta, indent=2))
    return meta


class TokenData:

    def __init__(self, data_dir, split):
        if split == "train":
            paths = sorted(p for p in os.listdir(data_dir) if p.startswith("train_") and p.endswith(".bin"))
        else:
            paths = ["val.bin"]
        self.arrays = [np.memmap(os.path.join(data_dir, p), dtype=np.uint16, mode="r") for p in paths]
        self.arrays = [a for a in self.arrays if len(a) > 0]
        self.lens = np.array([len(a) for a in self.arrays], dtype=np.float64)
        assert len(self.arrays), f"no {split} tokens in {data_dir}"

    @property
    def n_tokens(self):
        return int(self.lens.sum())

    def sample(self, rng, batch, length):
        p = self.lens / self.lens.sum()
        out = np.empty((batch, length), dtype=np.int64)
        for b in range(batch):
            a = self.arrays[rng.choice(len(self.arrays), p=p)]
            start = rng.integers(0, len(a) - length)
            out[b] = a[start:start + length]
        return torch.from_numpy(out)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)


class SwiGLU(nn.Module):
    def __init__(self, d, d_ff, dropout=0.0):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)
        self.w3 = nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))


def rope_tables(pos, head_dim, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=pos.device).float() / head_dim))
    f = pos.float()[..., None] * inv
    return f.cos()[:, None], f.sin()[:, None]


def apply_rope(x, cos, sin):
    h = x.shape[-1] // 2
    x1, x2 = x[..., :h].float(), x[..., h:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).type_as(x)


class Attention(nn.Module):

    def __init__(self, d, n_head, dropout=0.0, cross=False):
        super().__init__()
        self.h, self.hd, self.cross, self.dropout = n_head, d // n_head, cross, dropout
        self.wq = nn.Linear(d, d, bias=False)
        self.wk = nn.Linear(d, d, bias=False)
        self.wv = nn.Linear(d, d, bias=False)
        self.wo = nn.Linear(d, d, bias=False)

    def _split(self, t):
        B, T, _ = t.shape
        return t.view(B, T, self.h, self.hd).transpose(1, 2)

    def forward(self, x, rope=None, mask=None, causal=False, mem=None, cache=None):
        B, T, _ = x.shape
        q = self._split(self.wq(x))
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.wk(mem)), self._split(self.wv(mem))
                if cache is not None:
                    cache["k"], cache["v"] = k, v
        else:
            k, v = self._split(self.wk(x)), self._split(self.wv(x))
            if rope is not None:
                q, k = apply_rope(q, *rope), apply_rope(k, *rope)
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], 2)
                    v = torch.cat([cache["v"], v], 2)
                cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=causal and mask is None,
            dropout_p=self.dropout if self.training else 0.0)
        return self.wo(y.transpose(1, 2).reshape(B, T, -1))


class Block(nn.Module):
    def __init__(self, cfg, cross=False):
        super().__init__()
        d = cfg.d_model
        self.n1 = RMSNorm(d)
        self.attn = Attention(d, cfg.n_head, cfg.dropout)
        self.has_cross = cross
        if cross:
            self.nc = RMSNorm(d)
            self.xattn = Attention(d, cfg.n_head, cfg.dropout, cross=True)
        self.n2 = RMSNorm(d)
        self.mlp = SwiGLU(d, cfg.d_ff, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope, mask=None, causal=False, mem=None, mem_mask=None, cache=None):
        sc = None if cache is None else cache.setdefault("self", {})
        x = x + self.drop(self.attn(self.n1(x), rope, mask, causal, cache=sc))
        if self.has_cross:
            cc = None if cache is None else cache.setdefault("cross", {})
            x = x + self.drop(self.xattn(self.nc(x), mask=mem_mask, mem=mem, cache=cc))
        return x + self.mlp(self.n2(x))


def run_blocks(blocks, x, grad_ckpt, *args, caches=None, **kw):
    for i, b in enumerate(blocks):
        if grad_ckpt and b.training and caches is None:
            x = torch.utils.checkpoint.checkpoint(lambda t, b=b: b(t, *args, **kw), x, use_reentrant=False)
        else:
            x = b(x, *args, cache=None if caches is None else caches[i], **kw)
    return x


def init_weights(model, n_residual_layers):
    for name, p in model.named_parameters():
        if p.dim() >= 2:
            std = 0.02
            if name.endswith(("wo.weight", "w2.weight")):
                std = 0.02 / math.sqrt(2 * n_residual_layers)
            nn.init.normal_(p, 0.0, std)


def left_pad(seqs, pad=PAD):
    P = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), P), pad, dtype=torch.long)
    valid = torch.zeros((len(seqs), P), dtype=torch.bool)
    for i, s in enumerate(seqs):
        if len(s):
            ids[i, P - len(s):] = torch.tensor(s, dtype=torch.long)
            valid[i, P - len(s):] = True
    return ids, valid


def positions_from_valid(valid):
    return (valid.long().cumsum(1) - 1).clamp(min=0)


def ce(logits, labels, per_token=False):
    loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1),
                           ignore_index=-100, reduction="none" if per_token else "mean")
    return loss.view(labels.shape) if per_token else loss


class DecoderLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.unilm = cfg.arch == "unilm"
        if self.unilm:
            self.mode_emb = nn.Embedding(2, cfg.d_model)
            self.mlm_transform = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(),
                                               RMSNorm(cfg.d_model))
        init_weights(self, cfg.n_layer)
        self.lm_head.weight = self.emb.weight

    def hidden(self, ids, pos=None, mask=None, causal=True, mode=0, caches=None):
        B, T = ids.shape
        x = self.emb(ids)
        if self.unilm:
            x = x + self.mode_emb.weight[mode]
        if pos is None:
            pos = torch.arange(T, device=ids.device)[None].expand(B, T)
        rope = rope_tables(pos, self.cfg.d_model // self.cfg.n_head, self.cfg.rope_base)
        x = run_blocks(self.blocks, x, self.cfg.grad_ckpt, rope, mask, causal, caches=caches)
        return self.norm(x)

    def forward(self, input_ids, labels, mode=0, per_token=False):
        if mode == 0:
            return ce(self.lm_head(self.hidden(input_ids, causal=True, mode=0)), labels, per_token)
        h = self.hidden(input_ids, causal=False, mode=1)
        sel = labels != -100
        return ce(self.lm_head(self.mlm_transform(h[sel])), labels[sel])

    @torch.no_grad()
    def prefill(self, ids, valid):
        B, P = ids.shape
        pos = positions_from_valid(valid)
        eye = torch.eye(P, dtype=torch.bool, device=ids.device)
        causal = torch.ones(P, P, dtype=torch.bool, device=ids.device).tril()
        mask = (causal & (valid[:, None, :] | eye))[:, None]
        caches = [{} for _ in self.blocks]
        h = self.hidden(ids, pos, mask, causal=False, mode=0, caches=caches)
        return self.lm_head(h[:, -1]), {"caches": caches, "valid": valid, "pos": pos[:, -1]}

    @torch.no_grad()
    def step(self, tok, st):
        st["pos"] = st["pos"] + 1
        st["valid"] = torch.cat([st["valid"], torch.ones_like(st["valid"][:, :1])], 1)
        mask = st["valid"][:, None, None, :]
        h = self.hidden(tok[:, None], st["pos"][:, None], mask, causal=False, mode=0, caches=st["caches"])
        return self.lm_head(h[:, -1]), st


class EncDecLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.enc = nn.ModuleList(Block(cfg) for _ in range(cfg.n_enc_layer))
        self.enc_norm = RMSNorm(cfg.d_model)
        self.dec = nn.ModuleList(Block(cfg, cross=True) for _ in range(cfg.n_dec_layer))
        self.dec_norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        init_weights(self, cfg.n_enc_layer + cfg.n_dec_layer)
        self.lm_head.weight = self.emb.weight

    def _rope(self, pos):
        return rope_tables(pos, self.cfg.d_model // self.cfg.n_head, self.cfg.rope_base)

    def encode(self, ids, valid=None):
        B, S = ids.shape
        if valid is None or bool(valid.all()):
            pos, mask = torch.arange(S, device=ids.device)[None].expand(B, S), None
        else:
            pos, mask = positions_from_valid(valid), valid[:, None, None, :]
        x = run_blocks(self.enc, self.emb(ids), self.cfg.grad_ckpt, self._rope(pos), mask, False)
        return self.enc_norm(x)

    def decode(self, dec_ids, mem, mem_valid, pos=None, caches=None):
        B, T = dec_ids.shape
        if pos is None:
            pos = torch.arange(T, device=dec_ids.device)[None].expand(B, T)
        mem_mask = None if mem_valid is None or bool(mem_valid.all()) else mem_valid[:, None, None, :]
        causal = caches is None or "self" not in caches[0]
        x = run_blocks(self.dec, self.emb(dec_ids), self.cfg.grad_ckpt, self._rope(pos), None,
                       causal and T > 1, mem=mem, mem_mask=mem_mask, caches=caches)
        return self.lm_head(self.dec_norm(x))

    def forward(self, enc_ids, enc_valid, dec_in, labels, per_token=False):
        mem = self.encode(enc_ids, enc_valid)
        return ce(self.decode(dec_in, mem, enc_valid), labels, per_token)

    @torch.no_grad()
    def prefill(self, ids, valid):
        mem = self.encode(ids, valid)
        caches = [{} for _ in self.dec]
        start = torch.full((ids.size(0), 1), SEP, dtype=torch.long, device=ids.device)
        pos = torch.zeros_like(start)
        logits = self.decode(start, mem, valid, pos, caches)
        return logits[:, -1], {"caches": caches, "mem": mem, "valid": valid, "pos": pos[:, 0]}

    @torch.no_grad()
    def step(self, tok, st):
        st["pos"] = st["pos"] + 1
        logits = self.decode(tok[:, None], st["mem"], st["valid"], st["pos"][:, None], st["caches"])
        return logits[:, -1], st


class Retention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, H = cfg.d_model, cfg.n_head
        self.h, self.hd = H, d // H
        self.wq = nn.Linear(d, d, bias=False)
        self.wk = nn.Linear(d, d, bias=False)
        self.wv = nn.Linear(d, d, bias=False)
        self.wg = nn.Linear(d, d, bias=False)
        self.wo = nn.Linear(d, d, bias=False)
        self.gn_weight = nn.Parameter(torch.ones(d))
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5.0 - torch.arange(H).float())),
                             persistent=False)
        self._mask_cache = {}

    def decay_mask(self, T, device):
        key = (T, str(device))
        if key not in self._mask_cache:
            i = torch.arange(T, device=device).float()
            diff = i[:, None] - i[None, :]
            D = torch.exp(self.log_gamma.to(device)[:, None, None] * diff.clamp(min=0))
            D = D * (diff >= 0)
            D = D / D.sum(-1, keepdim=True).sqrt()
            self._mask_cache = {key: D}
        return self._mask_cache[key]

    def _qkv(self, x, rope):
        B, T, _ = x.shape
        sp = lambda t: t.view(B, T, self.h, self.hd).transpose(1, 2)
        q, k, v = sp(self.wq(x)), sp(self.wk(x)), sp(self.wv(x))
        q, k = apply_rope(q, *rope), apply_rope(k, *rope)
        return q, k * self.hd ** -0.5, v

    def _out(self, x, o):
        B, T = x.shape[:2]
        o = o.float()
        o = o / o.detach().abs().amax(-1, keepdim=True).clamp(min=1e-20)
        o = F.layer_norm(o, (self.hd,), eps=1e-6)
        o = o.transpose(1, 2).reshape(B, T, -1) * self.gn_weight.float()
        return self.wo(F.silu(self.wg(x)) * o.type_as(x))

    def forward(self, x, rope, valid=None, return_state=False):
        B, T, _ = x.shape
        q, k, v = self._qkv(x, rope)
        if valid is not None:
            m = valid[:, None, :, None].to(k.dtype)
            k, v = k * m, v * m
        D = self.decay_mask(T, x.device)
        qk = (q @ k.transpose(-1, -2)) * D.to(q.dtype)
        qk = qk / qk.detach().abs().sum(-1, keepdim=True).clamp(min=1)
        o = qk @ v
        state = None
        if return_state:
            w = torch.exp(self.log_gamma[:, None] * (T - 1 - torch.arange(T, device=x.device)).float())
            state = torch.einsum("bhtk,bhtv,ht->bhkv", k.float(), v.float(), w)
        return self._out(x, o), state

    def step(self, x, rope, S):
        q, k, v = self._qkv(x, rope)
        S = torch.exp(self.log_gamma)[None, :, None, None] * S + k.float().transpose(-1, -2) @ v.float()
        return self._out(x, q.float() @ S), S


class RetBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = RMSNorm(cfg.d_model)
        self.ret = Retention(cfg)
        self.n2 = RMSNorm(cfg.d_model)
        self.mlp = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope, valid=None, return_state=False):
        y, st = self.ret(self.n1(x), rope, valid, return_state)
        x = x + self.drop(y)
        return x + self.mlp(self.n2(x)), st

    def step(self, x, rope, S):
        y, S = self.ret.step(self.n1(x), rope, S)
        x = x + y
        return x + self.mlp(self.n2(x)), S


class RetNetLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList(RetBlock(cfg) for _ in range(cfg.n_layer))
        self.norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        init_weights(self, cfg.n_layer)
        self.lm_head.weight = self.emb.weight

    def _rope(self, pos):
        return rope_tables(pos, self.cfg.d_model // self.cfg.n_head, self.cfg.rope_base)

    def hidden(self, ids, valid=None, return_states=False):
        B, T = ids.shape
        pos = (torch.arange(T, device=ids.device)[None].expand(B, T) if valid is None
               else positions_from_valid(valid))
        rope = self._rope(pos)
        x, states = self.emb(ids), []
        for b in self.blocks:
            if self.cfg.grad_ckpt and self.training:
                x, st = torch.utils.checkpoint.checkpoint(b, x, rope, valid, False, use_reentrant=False)
            else:
                x, st = b(x, rope, valid, return_states)
            states.append(st)
        return self.norm(x), states, pos

    def forward(self, input_ids, labels, per_token=False):
        h, _, _ = self.hidden(input_ids)
        return ce(self.lm_head(h), labels, per_token)

    @torch.no_grad()
    def prefill(self, ids, valid):
        h, states, pos = self.hidden(ids, valid, return_states=True)
        return self.lm_head(h[:, -1]), {"S": states, "pos": pos[:, -1]}

    @torch.no_grad()
    def step(self, tok, st):
        st["pos"] = st["pos"] + 1
        rope = self._rope(st["pos"][:, None])
        x = self.emb(tok[:, None])
        for i, b in enumerate(self.blocks):
            x, st["S"][i] = b.step(x, rope, st["S"][i])
        return self.lm_head(self.norm(x)[:, -1]), st


def build_model(mcfg: ModelConfig):
    return {"gpt": DecoderLM, "unilm": DecoderLM, "encdec": EncDecLM, "retnet": RetNetLM}[mcfg.arch](mcfg)


def count_params(model):
    seen, n = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            n += p.numel()
    return n


def random_spans_mask(length, density, mean_span, rng):
    num_noise = max(1, int(round(length * density)))
    num_spans = max(1, min(N_SENTINELS - 1, int(round(num_noise / mean_span))))
    num_nonnoise = length - num_noise

    def partition(n_items, n_segs):
        cuts = np.sort(rng.choice(np.arange(1, n_items), n_segs - 1, replace=False)) if n_segs > 1 else []
        return np.diff(np.concatenate([[0], cuts, [n_items]])).astype(int)

    noise, keep = partition(num_noise, num_spans), partition(num_nonnoise, num_spans)
    mask, pos = np.zeros(length, dtype=bool), 0
    for a, b in zip(keep, noise):
        pos += a
        mask[pos:pos + b] = True
        pos += b
    return mask


def make_batch(arch, raw, rng, mcfg, train=True, vocab_size=None):
    T = mcfg.max_len
    if arch in ("gpt", "retnet"):
        return {"input_ids": raw[:, :T], "labels": raw[:, 1:T + 1]}, "clm"
    if arch == "unilm":
        if not train or rng.random() < 0.5:
            return {"input_ids": raw[:, :T], "labels": raw[:, 1:T + 1], "mode": 0}, "clm"
        x = raw[:, :T].clone()
        labels = torch.full_like(x, -100)
        sel = torch.from_numpy(rng.random(x.shape) < 0.15)
        labels[sel] = x[sel]
        r = torch.from_numpy(rng.random(x.shape))
        x[sel & (r < 0.8)] = MASK
        rand_pos = sel & (r >= 0.8) & (r < 0.9)
        x[rand_pos] = torch.from_numpy(
            rng.integers(N_SPECIAL, vocab_size or mcfg.vocab_size, size=int(rand_pos.sum()))).long()
        return {"input_ids": x, "labels": labels, "mode": 1}, "mlm"
    if arch == "encdec":
        E, Dl = mcfg.enc_len, mcfg.dec_len
        B = raw.size(0)
        if not train or rng.random() < 0.5:
            labels = raw[:, E:E + Dl]
            dec_in = torch.cat([torch.full((B, 1), SEP, dtype=torch.long), labels[:, :-1]], 1)
            return {"enc_ids": raw[:, :E], "enc_valid": torch.ones(B, E, dtype=torch.bool),
                    "dec_in": dec_in, "labels": labels}, "prefix"
        enc_ids = torch.full((B, E), PAD, dtype=torch.long)
        enc_valid = torch.zeros(B, E, dtype=torch.bool)
        labels = torch.full((B, Dl), -100, dtype=torch.long)
        dec_in = torch.full((B, Dl), PAD, dtype=torch.long)
        for b in range(B):
            toks = raw[b, :E].numpy()
            m = random_spans_mask(E, 0.15, 3.0, rng)
            enc, tgt, sid, prev = [], [], 0, False
            for t, is_noise in zip(toks, m):
                if is_noise:
                    if not prev:
                        enc.append(SENTINEL0 + sid)
                        tgt.append(SENTINEL0 + sid)
                        sid += 1
                    tgt.append(int(t))
                else:
                    enc.append(int(t))
                prev = is_noise
            tgt = (tgt + [EOS])[:Dl]
            enc_ids[b, :len(enc)] = torch.tensor(enc)
            enc_valid[b, :len(enc)] = True
            labels[b, :len(tgt)] = torch.tensor(tgt)
            dec_in[b, 0] = BOS
            dec_in[b, 1:len(tgt)] = torch.tensor(tgt[:-1])
        return {"enc_ids": enc_ids, "enc_valid": enc_valid, "dec_in": dec_in, "labels": labels}, "span"
    raise ValueError(arch)


def setup_device():
    world = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local = int(os.environ.get("LOCAL_RANK", "0"))
    if world > 1:
        torch.distributed.init_process_group("nccl" if torch.cuda.is_available() else "gloo")
    if torch.cuda.is_available():
        torch.cuda.set_device(local)
        device = torch.device("cuda", local)
    else:
        device = torch.device("cpu")
    return device, rank, world


def amp_dtype(device):
    if device.type != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16


def lr_at(step, total, cfg, start=0):
    peak = cfg.lr if start == 0 else cfg.lr * cfg.extend_peak_frac
    floor = cfg.lr * cfg.min_lr_frac
    step, total = step - start, total - start
    if step < cfg.warmup_steps:
        return floor * (start > 0) + (peak - floor * (start > 0)) * (step + 1) / cfg.warmup_steps
    prog = min(1.0, (step - cfg.warmup_steps) / max(1, total - cfg.warmup_steps))
    return floor + (peak - floor) * 0.5 * (1 + math.cos(math.pi * prog))


def to_dev(batch, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


@torch.no_grad()
def evaluate(model, val, cfg, mcfg, device, dtype):
    model.eval()
    rng = np.random.default_rng(1234)
    tot = tail = 0.0
    for _ in range(cfg.eval_batches):
        raw = val.sample(rng, cfg.micro_batch, mcfg.max_len + 1)
        batch, _ = make_batch(cfg.arch, raw, rng, mcfg, train=False)
        batch = to_dev(batch, device)
        with torch.autocast(device.type, dtype=dtype, enabled=dtype is not None):
            per_tok = model(**batch, per_token=True)
        tot += per_tok.mean().item()
        if cfg.arch == "encdec":
            tail += per_tok.mean().item()
        else:
            E, Dl = mcfg.enc_len, mcfg.dec_len
            tail += per_tok[:, E - 1:E - 1 + Dl].mean().item()
    model.train()
    return tot / cfg.eval_batches, tail / cfg.eval_batches


def save_atomic(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def export_weights(raw_model, mcfg, path, extra=None):
    sd = {k: v.detach().to("cpu", torch.float16 if v.is_floating_point() else v.dtype)
          for k, v in raw_model.state_dict().items()}
    save_atomic({"model_config": asdict(mcfg), "state_dict": sd, **(extra or {})}, path)


def load_model(path, device="cpu"):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    mcfg = ModelConfig(**ck["model_config"])
    model = build_model(mcfg)
    sd = ck.get("state_dict") or ck["model"]
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    return model.to(device).eval(), mcfg


def hf_push(cfg, files):
    if not cfg.hf_push_repo or int(os.environ.get("RANK", "0")) != 0:
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=os.environ.get("HF_TOKEN"))
        api.create_repo(cfg.hf_push_repo, private=True, exist_ok=True)
        for f in files:
            api.upload_file(path_or_fileobj=f, path_in_repo=f"{cfg.run_name}/{os.path.basename(f)}",
                            repo_id=cfg.hf_push_repo)
    except Exception as e:
        log0(f"[hf_push] failed: {e}")


def train(cfg):
    device, rank, world = setup_device()
    seed_everything(cfg.seed + rank)
    dtype = amp_dtype(device)
    os.makedirs(cfg.run_dir, exist_ok=True)
    with open(os.path.join(cfg.data_dir, "meta.json")) as f:
        vocab = json.load(f)["vocab_size"]
    mcfg = cfg.model_config(vocab)
    model = build_model(mcfg).to(device)
    n_params = count_params(model)
    log0(f"[train] arch={cfg.arch} run={cfg.run_name} params={n_params/1e6:.2f}M "
         f"device={device} world={world} amp={dtype}")
    if not cfg.smoke:
        assert 50e6 <= n_params <= 300e6, "Phase 1 rule: 50M-300M parameters"
    train_data, val_data = TokenData(cfg.data_dir, "train"), TokenData(cfg.data_dir, "val")
    log0(f"[train] train bin tokens={train_data.n_tokens/1e6:.1f}M val={val_data.n_tokens/1e6:.2f}M")

    decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": cfg.weight_decay},
                             {"params": no_decay, "weight_decay": 0.0}],
                            lr=cfg.lr, betas=(0.9, cfg.beta2), eps=1e-8,
                            fused=device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16)

    per_micro = cfg.micro_batch * cfg.seq_len * world
    accum = max(1, cfg.batch_tokens // per_micro)
    tokens_per_step = per_micro * accum
    total_steps = math.ceil(cfg.train_tokens / tokens_per_step)
    step, best_val, history = 0, float("inf"), []
    sched_start = 0

    resume = cfg.resume_from or os.path.join(cfg.run_dir, "last.pt")
    if os.path.exists(resume):
        ck = torch.load(resume, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optim"])
        scaler.load_state_dict(ck["scaler"])
        step, best_val, history = ck["step"], ck["best_val"], ck.get("history", [])
        log0(f"[train] resumed from {resume} at step {step}")
        sched_start = ck.get("sched_start", 0)
        if ck.get("total_steps") and ck["total_steps"] != total_steps:
            if total_steps > ck["total_steps"] and step >= ck["total_steps"]:
                sched_start = step
                log0(f"[train] extending finished run: {ck['total_steps']} -> {total_steps} steps, "
                     f"re-warming to lr {cfg.lr * cfg.extend_peak_frac:.2e}")
            else:
                log0(f"[train] warning: total_steps changed {ck['total_steps']} -> {total_steps}")

    raw_model = model
    if cfg.compile:
        model = torch.compile(model)
    if world > 1:
        model = torch.nn.parallel.DistributedDataParallel(
            model, device_ids=[device.index] if device.type == "cuda" else None,
            find_unused_parameters=cfg.arch == "unilm")
    log0(f"[train] accum={accum} tokens/step={tokens_per_step} total_steps={total_steps} "
         f"(~{total_steps*tokens_per_step/1e6:.0f}M tokens)")

    def checkpoint(final=False):
        if rank != 0:
            return
        state = {"model": raw_model.state_dict(), "optim": opt.state_dict(), "scaler": scaler.state_dict(),
                 "step": step, "best_val": best_val, "history": history, "total_steps": total_steps,
                 "sched_start": sched_start,
                 "model_config": asdict(mcfg), "run_config": asdict(cfg)}
        save_atomic(state, os.path.join(cfg.run_dir, "last.pt"))
        with open(os.path.join(cfg.run_dir, "history.json"), "w") as f:
            json.dump(history, f)
        files = [os.path.join(cfg.run_dir, "last.pt")]
        if final:
            export_weights(raw_model, mcfg, os.path.join(cfg.run_dir, "model_final.pt"),
                           {"step": step, "params": n_params})
            files.append(os.path.join(cfg.run_dir, "model_final.pt"))
        hf_push(cfg, files)

    t_start, t_log = time.time(), time.time()
    model.train()
    stopped_early = False
    while step < total_steps:
        lr = lr_at(step, total_steps, cfg, sched_start)
        for g in opt.param_groups:
            g["lr"] = lr
        losses = {}
        for micro in range(accum):
            rng = np.random.default_rng([cfg.seed, step, micro, rank])
            raw = train_data.sample(rng, cfg.micro_batch, mcfg.max_len + 1)
            batch, tag = make_batch(cfg.arch, raw, rng, mcfg, train=True, vocab_size=vocab)
            batch = to_dev(batch, device)
            sync = micro == accum - 1 or world == 1
            ctx = model.no_sync() if (world > 1 and not sync) else _nullctx()
            with ctx:
                with torch.autocast(device.type, dtype=dtype, enabled=dtype is not None):
                    loss = model(**batch)
                scaler.scale(loss / accum).backward()
            losses.setdefault(tag, []).append(loss.item())
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), cfg.grad_clip).item()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        step += 1

        if step % cfg.log_every == 0 or step == total_steps:
            dt = time.time() - t_log
            t_log = time.time()
            rec = {"step": step, "lr": lr, "gnorm": round(gnorm, 3),
                   "tokens": step * tokens_per_step,
                   "tok_per_s": round(cfg.log_every * tokens_per_step / max(dt, 1e-9)),
                   "elapsed_h": round((time.time() - t_start) / 3600, 3),
                   **{f"loss_{k}": round(float(np.mean(v)), 4) for k, v in losses.items()}}
            history.append(rec)
            log0(json.dumps(rec))
        if step % cfg.eval_every == 0 or step == total_steps:
            vl, vt = evaluate(raw_model, val_data, cfg, mcfg, device, dtype)
            history.append({"step": step, "val_loss": vl, "val_ppl": math.exp(min(vl, 20)),
                            "val_tail_loss": vt})
            log0(f"[eval] step {step}: val_loss={vl:.4f} ppl={math.exp(min(vl, 20)):.2f} tail={vt:.4f}")
            if vl < best_val and rank == 0:
                best_val = vl
                export_weights(raw_model, mcfg, os.path.join(cfg.run_dir, "model_best.pt"),
                               {"step": step, "val_loss": vl, "params": n_params})
        if step % cfg.save_every == 0:
            checkpoint()
        if (time.time() - t_start) / 3600 > cfg.max_hours:
            log0(f"[train] time budget {cfg.max_hours}h reached at step {step}; checkpointing. "
                 f"Re-run with resume_from pointing at last.pt to continue.")
            stopped_early = True
            break
    checkpoint(final=not stopped_early)
    if rank == 0:
        plot_history(history, os.path.join(cfg.run_dir, "loss_curve.png"), cfg.run_name)
    if world > 1:
        torch.distributed.destroy_process_group()
    return {"step": step, "best_val": best_val, "params": n_params, "finished": not stopped_early}


class _nullctx:
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


def plot_history(history, path, title):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    plt.figure(figsize=(9, 4.5))
    for key in ("loss_clm", "loss_mlm", "loss_prefix", "loss_span"):
        pts = [(h["step"], h[key]) for h in history if key in h]
        if pts:
            plt.plot(*zip(*pts), label=f"train {key[5:]}")
    pts = [(h["step"], h["val_loss"]) for h in history if "val_loss" in h]
    if pts:
        plt.plot(*zip(*pts), "o-", label="val")
    pts = [(h["step"], h["val_tail_loss"]) for h in history if "val_tail_loss" in h]
    if pts:
        plt.plot(*zip(*pts), "s--", label="val tail (comparable)")
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(f"Phase 1 {title}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


class NgramBlocker:

    def __init__(self, n, batch):
        self.n, self.seen, self.hist = n, [dict() for _ in range(batch)], [[] for _ in range(batch)]

    def banned(self, b):
        if self.n <= 0 or len(self.hist[b]) < self.n - 1:
            return ()
        return self.seen[b].get(tuple(self.hist[b][len(self.hist[b]) - self.n + 1:]), ())

    def add(self, b, tok):
        h = self.hist[b]
        h.append(tok)
        if self.n > 0 and len(h) >= self.n:
            self.seen[b].setdefault(tuple(h[-self.n:-1]), set()).add(h[-1])


def sample_next(logits, generated, blocker, cfg, gen):
    logits = logits.float()
    logits[:, :N_SPECIAL] = -float("inf")
    B = logits.size(0)
    if cfg.repetition_penalty != 1.0 and generated.numel():
        prev = torch.gather(logits, 1, generated)
        prev = torch.where(prev > 0, prev / cfg.repetition_penalty, prev * cfg.repetition_penalty)
        logits.scatter_(1, generated, prev)
    for b in range(B):
        ban = blocker.banned(b)
        if ban:
            logits[b, list(ban)] = -float("inf")
    if cfg.temperature <= 0:
        return logits.argmax(-1)
    logits = logits / cfg.temperature
    if cfg.top_k > 0:
        kth = torch.topk(logits, min(cfg.top_k, logits.size(-1)), -1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if 0 < cfg.top_p < 1:
        sl, si = torch.sort(logits, descending=True)
        cp = sl.softmax(-1).cumsum(-1)
        drop = cp - sl.softmax(-1) > cfg.top_p
        sl = sl.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = logits.softmax(-1)
    return torch.multinomial(probs, 1, generator=gen).squeeze(1)


@torch.no_grad()
def generate_continuations(model, mcfg, tok, prompts, cfg, device, dtype=None, seed=42):
    keep = mcfg.enc_len if mcfg.arch == "encdec" else max(mcfg.max_len - cfg.gen_max_new_tokens, mcfg.max_len // 2)
    encs = [e.ids[-keep:] for e in tok.encode_batch(prompts)]
    order = sorted(range(len(prompts)), key=lambda i: len(encs[i]))
    out = [""] * len(prompts)
    gen = torch.Generator(device=device).manual_seed(seed)
    for s in range(0, len(order), cfg.gen_batch):
        idx = order[s:s + cfg.gen_batch]
        ids, valid = left_pad([encs[i] for i in idx])
        ids, valid = ids.to(device), valid.to(device)
        blocker = NgramBlocker(cfg.no_repeat_ngram, len(idx))
        generated = torch.empty((len(idx), 0), dtype=torch.long, device=device)
        with torch.autocast(device.type, dtype=dtype, enabled=dtype is not None):
            logits, st = model.prefill(ids, valid)
            for t in range(cfg.gen_max_new_tokens):
                nxt = sample_next(logits, generated, blocker, cfg, gen)
                generated = torch.cat([generated, nxt[:, None]], 1)
                for b, tkn in enumerate(nxt.tolist()):
                    blocker.add(b, tkn)
                if t >= cfg.gen_target_words and t % 16 == 0:
                    texts = tok.decode_batch(generated.tolist())
                    if all(len(x.split()) > cfg.gen_target_words for x in texts):
                        break
                if t < cfg.gen_max_new_tokens - 1:
                    logits, st = model.step(nxt, st)
        for b, text in zip(idx, tok.decode_batch(generated.tolist())):
            words = text.split()[:cfg.gen_target_words]
            out[b] = " ".join(words) if words else "the"
        log0(f"[generate] {min(s + cfg.gen_batch, len(order))}/{len(order)}")
    return out


def resolve_checkpoint(cfg):
    if cfg.gen_checkpoint in ("best", "final"):
        p = os.path.join(cfg.run_dir, f"model_{cfg.gen_checkpoint}.pt")
        if not os.path.exists(p):
            p = os.path.join(cfg.run_dir, "model_best.pt")
        return p
    return cfg.gen_checkpoint


def write_prediction(preds, path, zip_path):
    for k, v in preds.items():
        assert isinstance(v, dict) and isinstance(v.get("text"), str) and v["text"].strip(), k
    with open(path, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")


def generate_stage(cfg, prompts_dict=None, out_name="prediction.json", zip_name=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = amp_dtype(device)
    tok = load_tokenizer(os.path.join(cfg.data_dir, "tokenizer.json"))
    ckpt = resolve_checkpoint(cfg)
    model, mcfg = load_model(ckpt, device)
    log0(f"[generate] {ckpt}: {count_params(model)/1e6:.2f}M params, arch {mcfg.arch}")
    if prompts_dict is None:
        prompts_dict = load_json_file(cfg, cfg.test_file)
    items = list(prompts_dict.items())
    if cfg.max_test_examples:
        items = items[:cfg.max_test_examples]
    t0 = time.time()
    texts = generate_continuations(model, mcfg, tok, [record_text(r) for _, r in items], cfg, device, dtype, cfg.seed)
    preds = {str(k): {"text": t} for (k, _), t in zip(items, texts)}
    path = os.path.join(cfg.run_dir, out_name)
    zip_path = os.path.join(cfg.run_dir, zip_name or f"submission_{cfg.run_name}.zip")
    write_prediction(preds, path, zip_path)
    log0(f"[generate] wrote {len(preds)} predictions to {path} and {zip_path} ({time.time()-t0:.0f}s)")
    return preds


def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge_l(p, r):
    pt, rt = p.split(), r.split()
    if not pt or not rt:
        return 0.0
    l = _lcs(pt, rt)
    if not l:
        return 0.0
    pp, rr = l / len(pt), l / len(rt)
    return 2 * pp * rr / (pp + rr)


def bleu4(preds, refs):
    from collections import Counter
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        p, r = p.split(), r.split()
        pl, rl = pl + len(p), rl + len(r)
        for n in range(1, 5):
            pn = Counter(tuple(p[i:i + n]) for i in range(len(p) - n + 1))
            rn = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
            totals[n - 1] += sum(pn.values())
            clipped[n - 1] += sum(min(c, rn.get(g, 0)) for g, c in pn.items())
    prec = [c / t if t else 0.0 for c, t in zip(clipped, totals)]
    if min(prec) <= 0 or pl == 0:
        return 0.0
    bp = 1.0 if pl > rl else math.exp(1 - rl / pl)
    return bp * math.exp(sum(math.log(x) for x in prec) / 4)


def pseudo_eval(cfg, use_bertscore=False):
    with open(os.path.join(cfg.data_dir, "pseudo_test.json")) as f:
        prompts = json.load(f)
    with open(os.path.join(cfg.data_dir, "pseudo_truth.json")) as f:
        truth = json.load(f)
    saved = cfg.max_test_examples
    cfg.max_test_examples = 0
    preds = generate_stage(cfg, prompts, out_name="pseudo_prediction.json", zip_name="pseudo_submission.zip")
    cfg.max_test_examples = saved
    ids = list(preds)
    res = {"n": len(ids),
           "rougeL_f1": float(np.mean([rouge_l(preds[i]["text"], truth[i]["text"]) for i in ids])),
           "bleu4": bleu4([preds[i]["text"] for i in ids], [truth[i]["text"] for i in ids])}
    if use_bertscore:
        import bert_score
        _, _, f1 = bert_score.score([preds[i]["text"] for i in ids], [truth[i]["text"] for i in ids], lang="en")
        res["bertscore_f1"] = float(f1.mean())
    with open(os.path.join(cfg.run_dir, "pseudo_eval.json"), "w") as f:
        json.dump(res, f, indent=2)
    log0(f"[pseudo_eval] {res}")
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=["prepare", "train", "generate", "pseudo_eval"])
    ap.add_argument("--config", required=True)
    a = ap.parse_args()
    cfg = load_run_config(a.config)
    if a.stage == "prepare":
        prepare(cfg)
    elif a.stage == "train":
        train(cfg)
    elif a.stage == "generate":
        generate_stage(cfg)
    else:
        pseudo_eval(cfg)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile ft_core.py
#!/usr/bin/env python3
"""CS781 Capstone, Phases 2-7: shared fine-tuning toolkit (Darsh Kedia, 250311).

References:
  [1] Hu et al. 2022, LoRA
  [2] Bihany, Patel, Modi 2025, LoRMA (Findings of ACL 2025)
  [3] Zhao et al. 2024, Tuning LayerNorm in Attention; Ben Zaken et al. 2022, BitFit
  [4] Hinton, Vinyals, Dean 2015, Knowledge Distillation; Sanh et al. 2019, DistilBERT
"""
import json
import math
import os
import random
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def device_and_amp():
    if not torch.cuda.is_available():
        return torch.device("cpu"), None
    dev = torch.device("cuda")
    return dev, (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)


def import_p1(code_dir=None):
    if code_dir and code_dir not in sys.path:
        sys.path.insert(0, code_dir)
    import cs781_p1
    return cs781_p1


def checkpoint_arch(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    return ck["model_config"]["arch"]


def _spread(n_have, n_keep):
    if n_keep >= n_have:
        return list(range(n_have))
    if n_keep == 1:
        return [n_have - 1]
    return sorted({round(i * (n_have - 1) / (n_keep - 1)) for i in range(n_keep)})


class Backbone(nn.Module):

    def __init__(self, ckpt, code_dir=None, keep_layers=None, bidir=False, grad_ckpt=False,
                 keep_decoder=False):
        super().__init__()
        P1 = import_p1(code_dir)
        model, mcfg = P1.load_model(ckpt, "cpu")
        self.arch, self.mcfg, self.bidir = mcfg.arch, mcfg, bidir
        self.d_model = mcfg.d_model
        stack = "enc" if self.arch == "encdec" else "blocks"
        n_have = len(getattr(model, stack))
        self.layers_kept = _spread(n_have, keep_layers) if keep_layers else list(range(n_have))
        if keep_layers:
            setattr(model, stack, nn.ModuleList(getattr(model, stack)[i] for i in self.layers_kept))
        if self.arch == "encdec" and not keep_decoder:
            model.dec = nn.ModuleList()
        if self.arch == "unilm":
            model.mlm_transform = nn.Identity()
        model.cfg.grad_ckpt = grad_ckpt
        self.model = model
        self.source_ckpt = os.path.abspath(ckpt)

    def forward(self, ids, valid):
        m, full = self.model, bool(valid.all())
        mask = None if full else valid[:, None, None, :]
        if self.arch == "unilm":
            return m.hidden(ids, mask=mask, causal=False, mode=1)
        if self.arch == "gpt":
            if self.bidir:
                return m.hidden(ids, mask=mask, causal=False)
            return m.hidden(ids, causal=True)
        if self.arch == "encdec":
            return m.encode(ids, valid)
        if self.arch == "retnet":
            return m.hidden(ids, None if full else valid)[0]
        raise ValueError(self.arch)


class LoRALinear(nn.Module):
    adapter = True

    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        out_f, in_f = base.weight.shape
        self.A = nn.Parameter(torch.empty(r, in_f))
        self.B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))

    def merged_weight(self):
        return self.base.weight + self.scale * self.B @ self.A


_ROLL_IDX = {}


def _roll_index(d, device):
    key = (d, str(device))
    if key not in _ROLL_IDX:
        a = torch.arange(d, device=device)
        _ROLL_IDX[key] = (a[None, :] - a[:, None]) % d
    return _ROLL_IDX[key]


class LoRMALinear(nn.Module):
    adapter = True

    def __init__(self, base: nn.Linear, r=8, alpha=16, variant="plus", dropout=0.0):
        super().__init__()
        self.base, self.r, self.variant = base, r, variant
        self.scale = alpha / r if variant == "plus" else 1.0
        d = base.weight.shape[0]
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        if variant == "pi":
            with torch.no_grad():
                self.A[0].zero_()
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0
        self.drop = nn.Dropout(dropout)

    def transform(self):
        M = self.B @ self.A
        if self.variant == "plus":
            return torch.eye(M.shape[0], device=M.device, dtype=M.dtype) + self.scale * M
        return M.gather(1, _roll_index(M.shape[0], M.device))

    def forward(self, x):
        h = self.base(x)
        if self.variant == "plus":
            return h + self.scale * F.linear(F.linear(self.drop(h), self.A.to(h.dtype)), self.B.to(h.dtype))
        return F.linear(self.drop(h), self.transform().to(h.dtype))

    def merged_weight(self):
        return self.transform() @ self.base.weight


NORM_KEYS = ("n1.weight", "n2.weight", "nc.weight", "norm.weight", "enc_norm.weight",
             "dec_norm.weight", "gn_weight")
DEFAULT_TARGETS = ("wq", "wk", "wv", "wo")
METHODS = ("full", "lora", "lorma_plus", "lorma_pi", "norms", "head")


def apply_peft(backbone, method, r=8, alpha=16, targets=DEFAULT_TARGETS, dropout=0.05,
               train_norms=False):
    if method not in METHODS:
        raise ValueError(f"method must be one of {METHODS}")
    for p in backbone.parameters():
        p.requires_grad = method == "full"
    n_wrapped = 0
    if method in ("lora", "lorma_plus", "lorma_pi"):
        for name, mod in list(backbone.named_modules()):
            for cname, child in list(mod.named_children()):
                if isinstance(child, nn.Linear) and cname in targets:
                    if method == "lora":
                        new = LoRALinear(child, r, alpha, dropout)
                    else:
                        new = LoRMALinear(child, r, alpha, "plus" if method == "lorma_plus" else "pi", dropout)
                    setattr(mod, cname, new)
                    n_wrapped += 1
        for n, p in backbone.named_parameters():
            if n.endswith((".A", ".B")):
                p.requires_grad = True
    if method == "norms" or (train_norms and method != "full"):
        for n, p in backbone.named_parameters():
            if n.endswith(NORM_KEYS) or n.endswith("mode_emb.weight"):
                p.requires_grad = True
    n_train = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in {id(p): p for p in backbone.parameters()}.values())
    return {"method": method, "wrapped_linears": n_wrapped, "backbone_trainable": n_train,
            "backbone_total": n_all}


def is_adapter_param(name):
    return name.endswith((".A", ".B"))


def param_groups(model, lr_backbone, lr_head, lr_adapter, weight_decay=0.01):
    groups = {"backbone": [], "adapter": [], "head": []}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("backbone."):
            groups["adapter" if is_adapter_param(n) else "backbone"].append(p)
        else:
            groups["head"].append(p)
    lrs = {"backbone": lr_backbone, "adapter": lr_adapter, "head": lr_head}
    return [{"params": ps, "lr": lrs[k], "weight_decay": 0.0 if k == "adapter" else weight_decay, "name": k}
            for k, ps in groups.items() if ps]


def warmup_linear(opt, total_steps, warmup_frac=0.06):
    warm = max(1, int(warmup_frac * total_steps))
    f = lambda s: (s + 1) / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


class AmpStep:

    def __init__(self, opt, amp_dtype, clip=1.0):
        self.opt, self.clip = opt, clip
        self.scaler = torch.amp.GradScaler("cuda") if amp_dtype == torch.float16 else None

    def __call__(self, loss, params):
        if self.scaler:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_(params, self.clip)
            self.scaler.step(self.opt)
            self.scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(params, self.clip)
            self.opt.step()
        self.opt.zero_grad(set_to_none=True)


def kd_loss(student_logits, teacher_logp, T=2.0, mask=None):
    t = F.softmax(teacher_logp.float() / T, -1)
    s = F.log_softmax(student_logits.float() / T, -1)
    kl = (t * (torch.log(t.clamp(min=1e-9)) - s)).sum(-1)
    if mask is not None:
        kl = kl[mask]
    return kl.mean() * T * T


def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep}


def save_trainable(model, path, meta):
    sd = {k: (v.half() if v.is_floating_point() else v) for k, v in trainable_state(model).items()}
    torch.save({"meta": meta, "state_dict": sd}, path)


def load_trainable(model, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    res = model.load_state_dict({k: v.float() if v.is_floating_point() else v
                                 for k, v in ck["state_dict"].items()}, strict=False)
    return ck["meta"], res


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def write_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False)

In [ ]:
%%writefile p5_lsi.py
#!/usr/bin/env python3
"""CS781 Capstone, Phase 5: Legal Statute Identification, LSI (Darsh Kedia, 250311).

References:
  [1] Paul et al. 2022, LeSICiN; Vats et al. 2023, ILSI
  [2] Chalkidis et al. 2019, hierarchical encoders for long legal documents
  [3] Hinton et al. 2015, Knowledge Distillation
  [4] Hu et al. 2022, LoRA; Bihany, Patel, Modi 2025, LoRMA
"""
import argparse
import glob
import json
import math
import os
import random
import re
import sys
import time
import zipfile
import zlib
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ft_core as C

PAD = 0
NL = 100
GRID = [round(0.02 * k, 2) for k in range(1, 48)]


def find(data, name):
    for p in (os.path.join(data, name), os.path.join(data, "Train", name), os.path.join(data, "Test", name)):
        if os.path.exists(p):
            return p
    hits = glob.glob(os.path.join(data, "**", name), recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(f"{name} under {data}")


def _labels(r):
    return sorted({int(x) for x in r["labels"]})


def load_raw(data, need_train=True):
    test = [{"id": str(r["id"]), "text": r["text"]} for r in json.load(open(find(data, "test.json"), encoding="utf-8"))]
    tids = {r["id"] for r in test}
    dev = [{"id": str(r["id"]), "text": r["text"], "labels": _labels(r)}
           for r in json.load(open(find(data, "dev.json"), encoding="utf-8")) if str(r["id"]) not in tids]
    dev_texts = {r["text"] for r in dev}
    train, info = [], {}
    if need_train:
        seen, n_raw, n_devdup = set(), 0, 0
        for r in json.load(open(find(data, "train.json"), encoding="utf-8")):
            n_raw += 1
            if r["text"] in dev_texts:
                n_devdup += 1
                continue
            if r["text"] in seen or str(r["id"]) in tids:
                continue
            seen.add(r["text"])
            train.append({"id": str(r["id"]), "text": r["text"], "labels": _labels(r)})
        info.update(train_raw=n_raw, train_dropped_dev_text=n_devdup)
    info.update(train=len(train), dev=len(dev), test=len(test))
    return {"train": train, "dev": dev, "test": test}, info


def clean_text(t):
    return re.sub(r"\s+", " ", t).strip()


def prep(args):
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(args.tokenizer)
    data, info = load_raw(args.data, need_train=not args.no_train)
    out = {"info": info, "keep_head": args.keep_head, "keep_tail": args.keep_tail,
           "tokenizer": os.path.abspath(args.tokenizer), "keep_all": args.keep_all,
           "cue_ids": sorted({tok.token_to_id(t) for t in ("SECTION", "ACT") if tok.token_to_id(t) is not None})}
    t0 = time.time()
    for split, docs in data.items():
        recs = {}
        for k in range(0, len(docs), 512):
            part = docs[k:k + 512]
            encs = tok.encode_batch([clean_text(d["text"]) for d in part], add_special_tokens=False)
            for d, e in zip(part, encs):
                ids = e.ids or [1]
                recs[d["id"]] = {"head": torch.tensor(ids[:args.keep_head], dtype=torch.int16),
                                 "tail": torch.tensor(ids[-args.keep_tail:], dtype=torch.int16),
                                 "n_tok": len(ids), "labels": d.get("labels")}
                if args.keep_all:
                    recs[d["id"]]["all"] = torch.tensor(ids[:args.keep_all], dtype=torch.int16)
        out[split] = recs
        print(f"prep {split}: {len(recs)} docs, {time.time() - t0:.0f}s", flush=True)
    lens = sorted(r["n_tok"] for r in out["dev"].values())
    out["info"]["dev_median_tokens"] = lens[len(lens) // 2]
    torch.save(out, args.cache)
    print(json.dumps(out["info"]))


def chunks_of(rec, chunk_len, head_chunks, tail_chunks):
    n_tok = rec["n_tok"]
    head = rec["head"].tolist()[:chunk_len * head_chunks]
    chunks = [head[i:i + chunk_len] for i in range(0, len(head), chunk_len)]
    pos = list(range(len(chunks)))
    covered = len(head)
    if tail_chunks and n_tok > covered:
        rest = min(n_tok - covered, chunk_len * tail_chunks)
        tail = rec["tail"].tolist()[-rest:] if rest <= len(rec["tail"]) else rec["tail"].tolist()
        tc, end = [], len(tail)
        while end > 0 and len(tc) < tail_chunks:
            s = max(0, end - chunk_len)
            tc.append(tail[s:end])
            end = s
        tc = tc[::-1]
        chunks += tc
        pos += [head_chunks + k for k in range(len(tc) - 1, -1, -1)]
    return chunks, pos


def cue_chunks(rec, chunk_len, n, cue_ids):
    ids = rec["all"]
    chunks = [ids[k:k + chunk_len] for k in range(0, len(ids), chunk_len)]
    if len(chunks) <= n:
        keep = list(range(len(chunks)))
    else:
        cue = torch.zeros(len(ids), dtype=torch.bool)
        for c in cue_ids:
            cue |= ids == c
        score = [int(cue[k:k + chunk_len].sum()) for k in range(0, len(ids), chunk_len)]
        rest = sorted(range(1, len(chunks)), key=lambda k: (-score[k], k))[:n - 1]
        keep = sorted([0] + rest)
    return [chunks[k].tolist() for k in keep], list(range(len(keep)))


def multi_hot(labels):
    y = torch.zeros(NL)
    if labels:
        y[torch.tensor(labels, dtype=torch.long)] = 1.0
    return y


def collate(recs, ids, args, teacher=None):
    chunks, pos, owner, ys, tl = [], [], [], [], []
    for b, i in enumerate(ids):
        if getattr(args, "select", "headtail") == "cue":
            ch, ps = cue_chunks(recs[i], args.chunk_len, args.head_chunks + args.tail_chunks, args.cue_ids)
        else:
            ch, ps = chunks_of(recs[i], args.chunk_len, args.head_chunks, args.tail_chunks)
        chunks += ch
        pos += ps
        owner += [b] * len(ch)
        ys.append(multi_hot(recs[i]["labels"]))
        if teacher is not None:
            tl.append(teacher[i])
    T = max(len(c) for c in chunks)
    x = torch.full((len(chunks), T), PAD, dtype=torch.long)
    valid = torch.zeros((len(chunks), T), dtype=torch.bool)
    for k, c in enumerate(chunks):
        x[k, :len(c)] = torch.tensor(c, dtype=torch.long)
        valid[k, :len(c)] = True
    out = {"ids": x, "valid": valid, "pos": torch.tensor(pos), "owner": torch.tensor(owner),
           "n_docs": len(ids), "y": torch.stack(ys)}
    if teacher is not None:
        out["teacher"] = torch.stack(tl).float()
    return out


class DocHead(nn.Module):

    def __init__(self, H, max_pos, hidden=384, dropout=0.1):
        super().__init__()
        self.pos = nn.Embedding(max_pos, H)
        nn.init.zeros_(self.pos.weight)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(H, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * hidden, 1)
        self.out = nn.Linear(4 * hidden, NL)

    def forward(self, v, pos, owner, n_docs):
        v = self.drop(v + self.pos(pos))
        counts = torch.bincount(owner, minlength=n_docs)
        K = int(counts.max())
        x = v.new_zeros(n_docs, K, v.shape[-1])
        slot = torch.cat([torch.arange(int(c), device=v.device) for c in counts])
        x[owner, slot] = v
        packed = nn.utils.rnn.pack_padded_sequence(x, counts.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.gru(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=K)
        mask = torch.arange(K, device=v.device)[None, :] < counts[:, None]
        a = self.att(h).squeeze(-1).masked_fill(~mask, -1e4).softmax(-1)
        att = (a[..., None] * h).sum(1)
        mx = h.masked_fill(~mask[..., None], -1e4).max(1).values
        return self.out(self.drop(torch.cat([att, mx], -1)))


class LSIModel(nn.Module):
    def __init__(self, backbone, max_pos, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = DocHead(backbone.d_model, max_pos, dropout=dropout)

    def chunk_vectors(self, ids, valid):
        h = self.backbone(ids, valid).float()
        m = valid[..., None].to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)

    def forward(self, b, device):
        v = self.chunk_vectors(b["ids"].to(device), b["valid"].to(device))
        return self.head(v, b["pos"].to(device), b["owner"].to(device), b["n_docs"])


def build(args):
    bb = C.Backbone(args.ckpt, args.code_dir, keep_layers=args.keep_layers or None,
                    bidir=args.bidir, grad_ckpt=args.grad_ckpt)
    peft = C.apply_peft(bb, args.method, args.rank, args.alpha, tuple(args.targets.split(",")),
                        train_norms=args.train_norms)
    return LSIModel(bb, args.head_chunks + args.tail_chunks, args.dropout), peft


def kd_bce(student_logits, teacher_logits, T=2.0):
    t = torch.sigmoid(teacher_logits.float() / T)
    return F.binary_cross_entropy_with_logits(student_logits.float() / T, t) * T * T


def f1_per_label(Y, P):
    tp = (Y & P).sum(0).double()
    fp = (~Y & P).sum(0).double()
    fn = (Y & ~P).sum(0).double()
    den = 2 * tp + fp + fn
    return torch.where(den > 0, 2 * tp / den.clamp(min=1), torch.zeros_like(den))


def macro_f1(Y, P):
    return float(f1_per_label(Y, P).mean())


def micro_f1(Y, P):
    tp = float((Y & P).sum())
    den = 2 * tp + float((~Y & P).sum()) + float((Y & ~P).sum())
    return 2 * tp / den if den else 0.0


def decide(prob, thr, min_one=True):
    P = prob > (thr if torch.is_tensor(thr) else float(thr))
    if min_one:
        empty = ~P.any(1)
        if empty.any():
            P[empty, prob[empty].argmax(1)] = True
    return P


def tune_global(Y, prob, min_one=True):
    return max((macro_f1(Y, decide(prob, t, min_one)), -abs(t - 0.5), t) for t in GRID)[2]


def tune_per_label(Y, prob):
    best = torch.full((NL,), 0.5)
    best_f = torch.full((NL,), -1.0)
    for t in GRID:
        f = f1_per_label(Y, prob > t).float()
        upd = f > best_f + 1e-9
        best[upd], best_f[upd] = t, f[upd]
    return best


def dev_matrix(recs, prob_dict):
    ids = [i for i in recs if i in prob_dict]
    Y = torch.stack([multi_hot(recs[i]["labels"]) for i in ids]).bool()
    prob = torch.stack([prob_dict[i].float() for i in ids])
    return ids, Y, prob


def choose_thresholds(recs, prob_dict):
    ids, Y, prob = dev_matrix(recs, prob_dict)
    half = torch.tensor([zlib.crc32(i.encode()) % 2 for i in ids])
    cross = {"global": [], "per_label": []}
    for a in (0, 1):
        A, B = half == a, half != a
        g = tune_global(Y[A], prob[A])
        pl = tune_per_label(Y[A], prob[A])
        cross["global"].append(macro_f1(Y[B], decide(prob[B], g)))
        cross["per_label"].append(macro_f1(Y[B], decide(prob[B], pl)))
    cg, cp = sum(cross["global"]) / 2, sum(cross["per_label"]) / 2
    g = tune_global(Y, prob)
    pl = tune_per_label(Y, prob)
    use_pl = cp > cg + 0.003
    thr = pl if use_pl else g
    P = decide(prob, thr)
    return {"rule": "per_label" if use_pl else "global", "thr": thr.tolist() if use_pl else g,
            "global_thr": g, "dev_f1_at_0.5": round(macro_f1(Y, decide(prob, 0.5)), 4),
            "dev_f1_cross_global": round(cg, 4), "dev_f1_cross_per_label": round(cp, 4),
            "dev_f1_honest": round(max(cg, cp) if use_pl else cg, 4),
            "dev_f1_tuned": round(macro_f1(Y, P), 4), "dev_micro_tuned": round(micro_f1(Y, P), 4),
            "dev_labels_per_doc_pred": round(float(P.sum(1).float().mean()), 3),
            "dev_labels_per_doc_gold": round(float(Y.sum(1).float().mean()), 3)}


def quick_dev_f1(recs, prob_dict):
    _, Y, prob = dev_matrix(recs, prob_dict)
    return macro_f1(Y, decide(prob, tune_global(Y, prob)))


def predictions(test_ids, prob_dict, thr):
    prob = torch.stack([prob_dict[i].float() for i in test_ids])
    thr = torch.tensor(thr) if isinstance(thr, list) else thr
    P = decide(prob, thr)
    return [{"id": i, "pred": [int(j) for j in torch.nonzero(P[k]).flatten().tolist()]} for k, i in enumerate(test_ids)]


def validate(test_ids, pred):
    ids = [p["id"] for p in pred]
    assert isinstance(pred, list) and pred, "prediction must be a non-empty list"
    assert len(ids) == len(set(ids)), "duplicate ids"
    assert set(ids) == set(test_ids), f"ids differ: missing {len(set(test_ids) - set(ids))}, extra {len(set(ids) - set(test_ids))}"
    n = 0
    for p in pred:
        assert isinstance(p["id"], str) and isinstance(p["pred"], list), p
        assert all(type(x) is int and 0 <= x < NL for x in p["pred"]), p
        assert len(p["pred"]) == len(set(p["pred"])), p
        n += len(p["pred"])
    return f"ok: {len(ids)} docs, {n / len(ids):.2f} labels per doc, {sum(not p['pred'] for p in pred)} empty"


def write_submission(pred, out_dir):
    pj = os.path.join(out_dir, "prediction.json")
    with open(pj, "w", encoding="utf-8") as f:
        json.dump(pred, f)
    with zipfile.ZipFile(os.path.join(out_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, "prediction.json")


def doc_tokens(rec, args):
    return min(rec["n_tok"], args.chunk_len * (args.head_chunks + args.tail_chunks))


def batches_by_length(recs, ids, args, batch_docs):
    budget = batch_docs * args.chunk_len * (args.head_chunks + args.tail_chunks)
    order = sorted(ids, key=lambda i: -doc_tokens(recs[i], args))
    out, cur = [], []
    for i in order:
        if cur and (doc_tokens(recs[cur[0]], args) * (len(cur) + 1) > budget or len(cur) >= 64):
            out.append(cur)
            cur = []
        cur.append(i)
    if cur:
        out.append(cur)
    return out


def train_batches(recs, ids, args, rng):
    budget = args.batch_docs * args.chunk_len * (args.head_chunks + args.tail_chunks)
    order = sorted(ids, key=lambda i: (doc_tokens(recs[i], args), rng.random()))
    out, cur, cur_max = [], [], 0
    for i in order:
        m = max(cur_max, doc_tokens(recs[i], args))
        if cur and (m * (len(cur) + 1) > budget or len(cur) >= args.max_batch_docs):
            out.append(cur)
            cur, m = [], doc_tokens(recs[i], args)
        cur.append(i)
        cur_max = m
    if cur:
        out.append(cur)
    rng.shuffle(out)
    return out


@torch.no_grad()
def doc_logits(model, recs, ids, args, device, amp):
    model.eval()
    out = {}
    for chunk in batches_by_length(recs, ids, args, args.eval_batch):
        b = collate(recs, chunk, args)
        with torch.autocast(device.type, dtype=amp, enabled=amp is not None):
            logits = model(b, device)
        lg = logits.float().cpu()
        for j, i in enumerate(chunk):
            out[i] = lg[j]
    return out


def to_prob(logits):
    return {k: torch.sigmoid(v).half() for k, v in logits.items()}


@torch.no_grad()
def chunk_features(model, recs, ids, args, device, amp, log):
    model.eval()
    feats, t0 = {}, time.time()
    bl = batches_by_length(recs, ids, args, args.eval_batch)
    for n, chunk in enumerate(bl):
        b = collate(recs, chunk, args)
        with torch.autocast(device.type, dtype=amp, enabled=amp is not None):
            v = model.chunk_vectors(b["ids"].to(device), b["valid"].to(device)).float().cpu()
        for j, i in enumerate(chunk):
            sel = b["owner"] == j
            feats[i] = (v[sel].half(), b["pos"][sel])
        if n % 50 == 0:
            log(f"  features {n + 1}/{len(bl)} batches, {time.time() - t0:.0f}s")
    return feats


def head_inputs(feats, ids, device):
    v = torch.cat([feats[i][0] for i in ids]).float().to(device)
    pos = torch.cat([feats[i][1] for i in ids]).to(device)
    owner = torch.cat([torch.full((len(feats[i][1]),), k) for k, i in enumerate(ids)]).to(device)
    return v, pos, owner


@torch.no_grad()
def head_logits(model, feats, ids, device, bs=256):
    model.eval()
    out = {}
    for k in range(0, len(ids), bs):
        chunk = ids[k:k + bs]
        v, pos, owner = head_inputs(feats, chunk, device)
        lg = model.head(v, pos, owner, len(chunk)).float().cpu()
        for j, i in enumerate(chunk):
            out[i] = lg[j]
    return out


def load_teacher(dirs):
    tot = None
    for d in dirs:
        t = torch.load(os.path.join(d, "train_logits.pt"))
        tot = t if tot is None else {k: tot[k] + t[k] for k in tot if k in t}
    return {k: v.float() / len(dirs) for k, v in tot.items()}


def train(args):
    C.set_seed(args.seed)
    rng = random.Random(args.seed)
    device, amp = C.device_and_amp()
    os.makedirs(args.out, exist_ok=True)
    log_f = open(os.path.join(args.out, "train.log"), "a")

    def log(*a):
        s = " ".join(str(x) for x in a)
        print(s, flush=True)
        log_f.write(s + "\n")
        log_f.flush()

    cache = torch.load(args.cache, weights_only=False)
    if args.select == "cue":
        assert cache.get("keep_all"), "--select cue needs a cache made with prep --keep_all N"
        args.cue_ids = cache["cue_ids"]
    tr, dv, te = cache["train"], cache["dev"], cache["test"]
    assert not set(dv) & set(te), "dev must not contain test docs"
    if args.full:
        tr = {**tr, **dv}
    pool = sorted(tr)
    teacher = load_teacher(args.teacher) if args.teacher else None
    if teacher is not None:
        pool = [i for i in pool if i in teacher]
    if args.limit_docs:
        pool = random.Random(0).sample(pool, min(args.limit_docs, len(pool)))
        dv = dict(list(dv.items())[:args.limit_docs])
    dev_ids = [] if args.full else sorted(dv)
    sel_ids = random.Random(7).sample(dev_ids, min(args.dev_select_docs, len(dev_ids))) if args.dev_select_docs else dev_ids
    test_ids = [r["id"] for r in json.load(open(find(args.data, "test.json"), encoding="utf-8"))]
    assert set(test_ids) == set(te), "cache does not match test.json"
    if args.limit_docs:
        test_ids = test_ids[:args.limit_docs]
    model, peft = build(args)
    model.to(device)
    n_tr = C.count_trainable(model)
    per_epoch = min(args.docs_per_epoch or len(pool), len(pool))
    pos_cnt = torch.zeros(NL)
    for i in pool:
        pos_cnt += multi_hot(tr[i]["labels"])
    pw = ((len(pool) - pos_cnt) / pos_cnt.clamp(min=1)).sqrt().clamp(1.0, args.max_pos_weight).to(device)
    if args.no_pos_weight:
        pw = None
    log(json.dumps({"arch": model.backbone.arch, "layers_kept": len(model.backbone.layers_kept), **peft,
                    "trainable_total": n_tr, "train_pool": len(pool), "docs_per_epoch": per_epoch,
                    "dev_docs": len(dev_ids), "dev_select_docs": len(sel_ids), "device": str(device),
                    "amp": str(amp), "max_tokens_per_doc": args.chunk_len * (args.head_chunks + args.tail_chunks),
                    "pos_weight_range": [round(float(pw.min()), 2), round(float(pw.max()), 2)] if pw is not None else None}))
    best, best_state, hist = -1.0, None, []
    feats = None
    allrecs = {**tr, **dv, **te}
    if args.method == "head":
        need = set(rng.sample(pool, per_epoch)) | set(dev_ids) | set(test_ids)
        pool = [i for i in pool if i in need]
        log(f"method head: caching frozen chunk features for {len(need)} docs")
        feats = chunk_features(model, allrecs, sorted(need), args, device, amp, log)
    groups = C.param_groups(model, args.lr, args.lr_head, args.lr_adapter, args.weight_decay)
    opt = torch.optim.AdamW(groups)
    stepper = C.AmpStep(opt, amp)
    params = [p for p in model.parameters() if p.requires_grad]

    def epoch_batches():
        ids = rng.sample(pool, per_epoch)
        if feats is not None:
            rng.shuffle(ids)
            return [ids[k:k + args.batch_docs] for k in range(0, len(ids), args.batch_docs)]
        return train_batches(tr, ids, args, rng)

    est = len(epoch_batches())
    steps = args.epochs * math.ceil(est / args.accum)
    sched = C.warmup_linear(opt, steps, args.warmup)

    def logits_for(ids):
        if feats is not None:
            return head_logits(model, feats, ids, device)
        return doc_logits(model, allrecs, ids, args, device, amp)

    for ep in range(1, args.epochs + 1):
        model.train()
        bl = epoch_batches()
        t0, tot, nb = time.time(), 0.0, 0
        for chunk in bl:
            tl = None
            if feats is not None:
                v, pos, owner = head_inputs(feats, chunk, device)
                logits = model.head(v, pos, owner, len(chunk))
                y = torch.stack([multi_hot(tr[i]["labels"]) for i in chunk]).to(device)
                if teacher is not None:
                    tl = torch.stack([teacher[i] for i in chunk]).to(device)
            else:
                b = collate(tr, chunk, args, teacher)
                y = b["y"].to(device)
                tl = b["teacher"].to(device) if teacher is not None else None
                with torch.autocast(device.type, dtype=amp, enabled=amp is not None):
                    logits = model(b, device)
            loss = F.binary_cross_entropy_with_logits(logits.float(), y, pos_weight=pw)
            if tl is not None:
                loss = (1 - args.kd_alpha) * loss + args.kd_alpha * kd_bce(logits, tl, args.kd_T)
            (stepper.scaler.scale(loss / args.accum) if stepper.scaler else loss / args.accum).backward()
            nb += 1
            tot += loss.item()
            if nb % args.accum == 0 or nb == len(bl):
                if stepper.scaler:
                    stepper.scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(params, 1.0)
                    stepper.scaler.step(opt)
                    stepper.scaler.update()
                else:
                    nn.utils.clip_grad_norm_(params, 1.0)
                    opt.step()
                opt.zero_grad(set_to_none=True)
                sched.step()
            if nb % args.log_every == 0:
                log(f"  ep {ep} batch {nb}/{len(bl)} loss {tot / nb:.4f} {time.time() - t0:.0f}s")
        row = {"epoch": ep, "loss": round(tot / max(nb, 1), 4), "sec": round(time.time() - t0, 1)}
        if sel_ids:
            f1 = quick_dev_f1(dv, to_prob(logits_for(sel_ids)))
            row["dev_sel_f1"] = round(f1, 4)
            if f1 > best:
                best, best_state = f1, C.trainable_state(model)
        hist.append(row)
        log(json.dumps(row))
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
    run = {"args": vars(args), "peft": peft, "trainable": n_tr, "history": hist, "arch": model.backbone.arch,
           "layers_kept": model.backbone.layers_kept, "data": cache["info"]}
    thr = 0.5
    if dev_ids:
        t0 = time.time()
        dprob = to_prob(logits_for(dev_ids))
        torch.save(dprob, os.path.join(args.out, "dev_prob.pt"))
        ch = choose_thresholds(dv, dprob)
        thr = ch["thr"]
        run.update({k: v for k, v in ch.items() if k != "thr"}, threshold=thr)
        log(f"dev Macro-F1: honest (cross-half) {ch['dev_f1_honest']}, tuned {ch['dev_f1_tuned']} "
            f"[{ch['rule']}], at 0.5 {ch['dev_f1_at_0.5']} ({time.time() - t0:.0f}s)")
    t0 = time.time()
    tprob = to_prob(logits_for(test_ids))
    torch.save(tprob, os.path.join(args.out, "test_prob.pt"))
    pred = predictions(test_ids, tprob, thr)
    log(validate(test_ids, pred) + f" ({time.time() - t0:.0f}s)")
    write_submission(pred, args.out)
    if args.save_train_logits:
        ids = sorted(cache["train"])
        ids = random.Random(1234).sample(ids, min(args.save_train_logits, len(ids)))
        if feats is not None:
            ids = [i for i in ids if i in feats]
        log(f"teacher: logits on {len(ids)} train docs")
        tl = logits_for(ids)
        torch.save({k: v.half() for k, v in tl.items()}, os.path.join(args.out, "train_logits.pt"))
    if args.save_model:
        C.save_trainable(model, os.path.join(args.out, "model.pt"), run)
    C.write_json(run, os.path.join(args.out, "run.json"))
    log(f"wrote {args.out}/submission.zip")


def dev_labels(data):
    data_raw, _ = load_raw(data, need_train=False)
    return {r["id"]: {"labels": r["labels"]} for r in data_raw["dev"]}, [r["id"] for r in data_raw["test"]]


def load_probs(run, name):
    p = os.path.join(run, name)
    return torch.load(p) if os.path.exists(p) else None


def ensemble(args):
    dv, test_ids = dev_labels(args.data)
    w = [float(x) for x in args.weights.split(",")] if args.weights else [1.0] * len(args.runs)
    tp, dp = None, None
    for r, wi in zip(args.runs, w):
        t = load_probs(r, "test_prob.pt")
        tp = {k: wi * t[k].float() for k in t} if tp is None else {k: tp[k] + wi * t[k].float() for k in tp}
        d = load_probs(r, "dev_prob.pt")
        if dp is not False and d is not None:
            dp = {k: wi * d[k].float() for k in d} if dp is None else {k: dp[k] + wi * d[k].float() for k in dp if k in d}
        else:
            dp = False
    s = sum(w)
    tp = {k: (v / s).half() for k, v in tp.items()}
    os.makedirs(args.out, exist_ok=True)
    info, thr = {"runs": args.runs, "weights": w}, 0.5
    if dp:
        dp = {k: (v / s).half() for k, v in dp.items()}
        ch = choose_thresholds(dv, dp)
        thr = ch["thr"]
        info.update({k: v for k, v in ch.items() if k != "thr"}, threshold=thr)
        torch.save(dp, os.path.join(args.out, "dev_prob.pt"))
    pred = predictions(test_ids, tp, thr)
    print(validate(test_ids, pred))
    write_submission(pred, args.out)
    torch.save(tp, os.path.join(args.out, "test_prob.pt"))
    C.write_json(info, os.path.join(args.out, "run.json"))
    print(json.dumps({k: v for k, v in info.items() if k != "threshold"}))


def baseline(args):
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.multiclass import OneVsRestClassifier
    data, info = load_raw(args.data)
    sl = slice(args.char_start, args.char_end)
    T = lambda s: [clean_text(r["text"])[sl] for r in s]
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=400000, sublinear_tf=True,
                          dtype=np.float32, token_pattern=r"(?u)<\w+>|\b\w\w+\b")
    Xtr = vec.fit_transform(T(data["train"]))
    Xdv, Xte = vec.transform(T(data["dev"])), vec.transform(T(data["test"]))
    for m in (Xtr, Xdv, Xte):
        m.sort_indices()
    ytr = np.stack([multi_hot(r["labels"]).numpy() for r in data["train"]]).astype(np.uint8)
    clf = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=args.C, class_weight="balanced"),
                              n_jobs=args.jobs).fit(Xtr, ytr)
    os.makedirs(args.out, exist_ok=True)
    dprob = {r["id"]: torch.tensor(p).half() for r, p in zip(data["dev"], clf.predict_proba(Xdv))}
    tprob = {r["id"]: torch.tensor(p).half() for r, p in zip(data["test"], clf.predict_proba(Xte))}
    recs = {r["id"]: r for r in data["dev"]}
    ch = choose_thresholds(recs, dprob)
    torch.save(dprob, os.path.join(args.out, "dev_prob.pt"))
    torch.save(tprob, os.path.join(args.out, "test_prob.pt"))
    pred = predictions([r["id"] for r in data["test"]], tprob, ch["thr"])
    print(validate([r["id"] for r in data["test"]], pred))
    write_submission(pred, args.out)
    run = {"model": "tfidf_ovr_lr (REFERENCE ONLY, do not submit)", "C": args.C, "chars": [args.char_start, args.char_end],
           "features": Xtr.shape[1], "data": info, **{k: v for k, v in ch.items() if k != "thr"}, "threshold": ch["thr"]}
    C.write_json(run, os.path.join(args.out, "run.json"))
    print(json.dumps({k: v for k, v in run.items() if k != "threshold"}))


def score_file(data, path):
    dv, _ = dev_labels(data)
    if os.path.isdir(path):
        ch = choose_thresholds(dv, torch.load(os.path.join(path, "dev_prob.pt")))
        print(json.dumps({k: v for k, v in ch.items() if k != "thr"}))
        return
    pred = {p["id"]: p["pred"] for p in json.load(open(path))}
    ids = [i for i in dv if i in pred]
    Y = torch.stack([multi_hot(dv[i]["labels"]) for i in ids]).bool()
    P = torch.stack([multi_hot(pred[i]) for i in ids]).bool()
    print(f"dev docs scored: {len(ids)}  Macro-F1 {macro_f1(Y, P):.6f}  Micro-F1 {micro_f1(Y, P):.6f}  "
          f"subset acc {float((Y == P).all(1).float().mean()):.6f}")


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("prep")
    p.add_argument("--data", required=True)
    p.add_argument("--tokenizer", required=True)
    p.add_argument("--cache", required=True)
    p.add_argument("--keep_head", type=int, default=3072)
    p.add_argument("--keep_tail", type=int, default=1024)
    p.add_argument("--keep_all", type=int, default=0, help="also keep the first N tokens of every doc (for --select cue)")
    p.add_argument("--no_train", action="store_true", help="skip train.json (smoke tests)")
    t = sub.add_parser("train")
    t.add_argument("--data", required=True)
    t.add_argument("--cache", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--ckpt", required=True, help="Phase 1 export (model_best.pt) of any arch")
    t.add_argument("--code_dir", default=None, help="folder holding cs781_p1.py")
    t.add_argument("--method", default="full", choices=C.METHODS)
    t.add_argument("--rank", type=int, default=16)
    t.add_argument("--alpha", type=float, default=32)
    t.add_argument("--targets", default=",".join(C.DEFAULT_TARGETS))
    t.add_argument("--train_norms", action="store_true")
    t.add_argument("--bidir", action="store_true", help="gpt: drop the causal mask")
    t.add_argument("--keep_layers", type=int, default=0, help="student: keep this many evenly spaced layers")
    t.add_argument("--chunk_len", type=int, default=512)
    t.add_argument("--head_chunks", type=int, default=3, help="chunks from the start of the document")
    t.add_argument("--tail_chunks", type=int, default=1, help="chunks aligned to the end (not overlapping the head)")
    t.add_argument("--select", default="headtail", choices=["headtail", "cue"],
                   help="cue: first chunk + the chunks with the most <SECTION>/<ACT> markers (needs prep --keep_all)")
    t.add_argument("--dropout", type=float, default=0.1)
    t.add_argument("--epochs", type=int, default=2)
    t.add_argument("--docs_per_epoch", type=int, default=16000, help="fresh random sample each epoch (0 = all)")
    t.add_argument("--batch_docs", type=int, default=4, help="token budget per batch = batch_docs x max doc tokens")
    t.add_argument("--max_batch_docs", type=int, default=32)
    t.add_argument("--accum", type=int, default=4)
    t.add_argument("--eval_batch", type=int, default=8)
    t.add_argument("--dev_select_docs", type=int, default=2500, help="dev docs used for per-epoch selection (0 = all)")
    t.add_argument("--lr", type=float, default=3e-5, help="backbone LR (full / norms)")
    t.add_argument("--lr_adapter", type=float, default=5e-4, help="LoRA / LORMA LR")
    t.add_argument("--lr_head", type=float, default=1e-3)
    t.add_argument("--weight_decay", type=float, default=0.01)
    t.add_argument("--warmup", type=float, default=0.06)
    t.add_argument("--max_pos_weight", type=float, default=10.0)
    t.add_argument("--no_pos_weight", action="store_true")
    t.add_argument("--teacher", action="append", help="run dir(s) with train_logits.pt (distillation)")
    t.add_argument("--kd_alpha", type=float, default=0.5)
    t.add_argument("--kd_T", type=float, default=2.0)
    t.add_argument("--full", action="store_true", help="train on train+dev, no dev selection, threshold 0.5")
    t.add_argument("--save_train_logits", type=int, default=0, help="teacher: logits on this many train docs")
    t.add_argument("--save_model", action="store_true")
    t.add_argument("--grad_ckpt", action="store_true")
    t.add_argument("--log_every", type=int, default=500)
    t.add_argument("--seed", type=int, default=42)
    t.add_argument("--limit_docs", type=int, default=0, help="smoke tests only")
    e = sub.add_parser("ensemble")
    e.add_argument("--data", required=True)
    e.add_argument("--out", required=True)
    e.add_argument("--weights", default="")
    e.add_argument("runs", nargs="+")
    s = sub.add_parser("score")
    s.add_argument("--data", required=True)
    s.add_argument("pred")
    v = sub.add_parser("validate")
    v.add_argument("--data", required=True)
    v.add_argument("pred")
    b = sub.add_parser("baseline")
    b.add_argument("--data", required=True)
    b.add_argument("--out", required=True)
    b.add_argument("--C", type=float, default=4.0)
    b.add_argument("--char_start", type=int, default=None)
    b.add_argument("--char_end", type=int, default=40000)
    b.add_argument("--jobs", type=int, default=4)
    args = ap.parse_args(argv)
    if args.cmd == "prep":
        prep(args)
    elif args.cmd == "train":
        train(args)
    elif args.cmd == "ensemble":
        ensemble(args)
    elif args.cmd == "score":
        score_file(args.data, args.pred)
    elif args.cmd == "baseline":
        baseline(args)
    else:
        _, test_ids = dev_labels(args.data)
        print(validate(test_ids, json.load(open(args.pred))))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile p5_post.py
#!/usr/bin/env python3
"""CS781 Phase 5 (LSI): label-structure post-processing on saved probabilities (Darsh Kedia, 250311).

References:
  [1] Godbole & Sarawagi 2004, Discriminative Methods for Multi-labeled Classification
"""
import argparse
import json
import os
import sys
import zlib

import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import p5_lsi as P5

NL = P5.NL
LAMS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
CS = [round(0.6 + 0.05 * k, 2) for k in range(17)]


def load_avg(runs, name):
    tot = None
    for r in runs:
        d = torch.load(os.path.join(r, name))
        tot = {k: v.float() for k, v in d.items()} if tot is None else {k: tot[k] + d[k].float() for k in tot}
    return {k: v / len(runs) for k, v in tot.items()}


def cooc_matrix(data):
    raw, _ = P5.load_raw(data)
    Y = torch.stack([P5.multi_hot(r["labels"]) for r in raw["train"]])
    co = Y.T @ Y
    return co / co.diag().clamp(min=1)[:, None]


def apply_cooc(Pm, M, lam):
    prior = (Pm @ M) / Pm.sum(1, keepdim=True).clamp(min=1e-6)
    return ((1 - lam) * Pm + lam * prior.clamp(max=1.0)).clamp(0, 1)


def topk_decide(Pm, c):
    k = (c * Pm.sum(1)).round().clamp(1, 15).long()
    order = Pm.argsort(1, descending=True)
    out = torch.zeros_like(Pm, dtype=torch.bool)
    ranks = torch.arange(NL)[None, :] < k[:, None]
    out.scatter_(1, order, ranks)
    return out


def logit(Pm):
    p = Pm.clamp(1e-4, 1 - 1e-4)
    return (p / (1 - p)).log()


def fit_stack(Y, Pm, C=0.5):
    from sklearn.linear_model import LogisticRegression
    X = logit(Pm).numpy()
    W = torch.zeros(NL, NL)
    b = torch.zeros(NL)
    for j in range(NL):
        y = Y[:, j].numpy()
        if y.min() == y.max():
            b[j] = -10.0 if y.max() == 0 else 10.0
            continue
        m = LogisticRegression(C=C, max_iter=300).fit(X, y)
        W[j] = torch.tensor(m.coef_[0], dtype=torch.float32)
        b[j] = float(m.intercept_[0])
    return W, b


def apply_stack(params, Pm):
    W, b = params
    return torch.sigmoid(logit(Pm) @ W.T + b)


def fit_resid(Y, Pm, l2=1e-3, steps=300):
    X = logit(Pm)
    Yf = Y.float()
    W = torch.zeros(NL, NL, requires_grad=True)
    a = torch.ones(NL, requires_grad=True)
    b = torch.zeros(NL, requires_grad=True)
    off = 1 - torch.eye(NL)
    pos = Yf.sum(0)
    pw = ((len(Yf) - pos) / pos.clamp(min=1)).sqrt().clamp(1.0, 10.0)
    opt = torch.optim.Adam([W, a, b], lr=0.01)
    for _ in range(steps):
        s = X * a + X @ (W * off).T + b
        loss = torch.nn.functional.binary_cross_entropy_with_logits(s, Yf, pos_weight=pw) + l2 * (W * off).pow(2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return (W * off).detach(), a.detach(), b.detach()


def apply_resid(params, Pm):
    W, a, b = params
    X = logit(Pm)
    return torch.sigmoid(X * a + X @ W.T + b)


def halves(ids):
    return torch.tensor([zlib.crc32(i.encode()) % 2 for i in ids])


def bagged(Y, Pm, fit, apply):
    f = torch.arange(len(Y)) % 2
    out = torch.zeros_like(Pm)
    fits = []
    for a in (0, 1):
        tr, te = f != a, f == a
        fits.append(fit(Y[tr], Pm[tr]))
        out[te] = apply(fits[-1], Pm[te])
    return out, (lambda Q: sum(apply(p, Q) for p in fits) / len(fits))


def fit_variant(name, Y, Pm, M):
    if name == "base":
        thr = P5.tune_global(Y, Pm)
        return (lambda Q: Q), (lambda Q: P5.decide(Q, thr)), {"thr": thr}
    if name == "cooc":
        best = max((P5.macro_f1(Y, P5.decide(apply_cooc(Pm, M, l), P5.tune_global(Y, apply_cooc(Pm, M, l)))), -l, l)
                   for l in LAMS)
        lam = best[2]
        thr = P5.tune_global(Y, apply_cooc(Pm, M, lam))
        return (lambda Q: apply_cooc(Q, M, lam)), (lambda Q: P5.decide(Q, thr)), {"lam": lam, "thr": thr}
    if name == "topk":
        c = max((P5.macro_f1(Y, topk_decide(Pm, c)), -abs(c - 1), c) for c in CS)[2]
        return (lambda Q: Q), (lambda Q: topk_decide(Q, c)), {"c": c}
    if name.split("+")[0] in ("resid", "stack"):
        fit, apply = (fit_resid, apply_resid) if name.startswith("resid") else (fit_stack, apply_stack)
        oof, tf = bagged(Y, Pm, fit, apply)
        if "+topk" not in name:
            thr = P5.tune_global(Y, oof)
            return tf, (lambda Q: P5.decide(Q, thr)), {"thr": thr}
        c = max((P5.macro_f1(Y, topk_decide(oof, c)), -abs(c - 1), c) for c in CS)[2]
        return tf, (lambda Q: topk_decide(Q, c)), {"c": c}
    raise ValueError(name)


VARIANTS = ["base", "cooc", "topk", "resid", "resid+topk", "stack"]


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--variants", default=",".join(VARIANTS))
    ap.add_argument("runs", nargs="+")
    a = ap.parse_args(argv)
    dv, test_ids = P5.dev_labels(a.data)
    ids, Y, Pm = P5.dev_matrix(dv, load_avg(a.runs, "dev_prob.pt"))
    tp = load_avg(a.runs, "test_prob.pt")
    Pt = torch.stack([tp[i] for i in test_ids])
    M = cooc_matrix(a.data)
    h = halves(ids)
    os.makedirs(a.out, exist_ok=True)
    table = {}
    for name in a.variants.split(","):
        mac, mic, card = [], [], []
        for s in (0, 1):
            A, B = h == s, h != s
            tf, dec, _ = fit_variant(name, Y[A], Pm[A], M)
            PB = dec(tf(Pm[B]))
            mac.append(P5.macro_f1(Y[B], PB))
            mic.append(P5.micro_f1(Y[B], PB))
            card.append(float(PB.sum(1).float().mean()))
        tf, dec, params = fit_variant(name, Y, Pm, M)
        PT = dec(tf(Pt))
        pred = [{"id": i, "pred": [int(j) for j in torch.nonzero(PT[k]).flatten().tolist()]} for k, i in enumerate(test_ids)]
        d = os.path.join(a.out, name.replace("+", "_"))
        os.makedirs(d, exist_ok=True)
        msg = P5.validate(test_ids, pred)
        P5.write_submission(pred, d)
        table[name] = {"dev_f1_honest": round(sum(mac) / 2, 4), "dev_micro_honest": round(sum(mic) / 2, 4),
                       "labels_per_doc": round(sum(card) / 2, 2), "params_full_dev": params}
        json.dump({"runs": a.runs, "variant": name, **table[name]}, open(os.path.join(d, "run.json"), "w"), indent=1)
        print(f"{name:11s} honest Macro-F1 {table[name]['dev_f1_honest']:.4f}  Micro {table[name]['dev_micro_honest']:.4f}  "
              f"labels/doc {table[name]['labels_per_doc']:.2f}  {params}  test: {msg}", flush=True)
    json.dump({"runs": a.runs, "gold_labels_per_doc": round(float(Y.sum(1).float().mean()), 2), "table": table},
              open(os.path.join(a.out, "ablation.json"), "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
N = 600 if QUICK else DOCS_PER_EPOCH
E, EP = (1, 1) if QUICK else (2, 3)
COMMON = f"--docs_per_epoch {N}" + (" --dev_select_docs 300" if QUICK else "")
PLAN = [
    ("unilm_full_cue",  "unilm", f"--method full --epochs {E} --select cue", []),
    ("unilm_lorma_cue", "unilm", f"--method lorma_plus --rank 16 --alpha 32 --epochs {EP} --select cue", []),
]
PLAN = [p for p in PLAN if p[1] in CKPT]
for p in PLAN: print(p[0], p[1], p[2])

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
tokn = UserSecretsClient().get_secret("HF").strip().strip("<>")
root = snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                         allow_patterns=["Phase_5_LSI/*"], token=tokn, local_dir=f"{WORK}/hf")
DATA = f"{root}/Phase_5_LSI"
print(sorted(glob.glob(f"{DATA}/*/*")))
import subprocess
subprocess.run(["python", "p5_lsi.py", "prep", "--data", DATA, "--tokenizer", TOKENIZER, "--cache", CACHE, "--keep_all", "16384"], check=True)

In [ ]:
import subprocess, time, json
os.makedirs(OUT, exist_ok=True)
n_gpu = max(1, torch.cuda.device_count())
pending, running, done, failed = list(PLAN), {}, set(), set()
t_start = time.time()
while pending or running:
    for g in range(n_gpu):
        if g in running:
            continue
        job = next((p for p in pending if all(d in done for d in p[3])), None)
        if job is None:
            continue
        if time.time() - t_start > MAX_HOURS * 3600:
            print("time budget reached, skipping:", [p[0] for p in pending]); pending.clear(); break
        pending.remove(job)
        name, arch, extra, _ = job
        cmd = (f"python p5_lsi.py train --data {DATA} --cache {CACHE} --out {OUT}/{name} "
               f"--ckpt {CKPT[arch]} --code_dir . {COMMON} {extra}")
        os.makedirs(f"{OUT}/{name}", exist_ok=True)
        log = open(f"{OUT}/{name}/stdout.log", "w")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        running[g] = (job, subprocess.Popen(cmd.split(), stdout=log, stderr=subprocess.STDOUT, env=env), time.time())
        print(f"[{(time.time()-t_start)/60:5.1f} min] GPU{g} start {name}: {extra}", flush=True)
    for g, (job, proc, t0) in list(running.items()):
        if proc.poll() is not None:
            del running[g]
            ok = proc.returncode == 0 and os.path.exists(f"{OUT}/{job[0]}/submission.zip")
            (done if ok else failed).add(job[0])
            tail = open(f"{OUT}/{job[0]}/stdout.log").read().strip().splitlines()[-3:]
            print(f"[{(time.time()-t_start)/60:5.1f} min] GPU{g} {'done' if ok else 'FAILED'} {job[0]} "
                  f"({(time.time()-t0)/60:.1f} min)\n    " + "\n    ".join(tail), flush=True)
    for p in list(pending):
        if any(d in failed for d in p[3]):
            pending.remove(p); failed.add(p[0]); print("skipped", p[0], "(dependency failed)")
    time.sleep(15)
print("done:", sorted(done), "failed:", sorted(failed))

In [ ]:
import shutil, subprocess, sys
sys.path.insert(0, ".")
import p5_lsi as P5
for f in glob.glob("/kaggle/input/**/p5/*/dev_prob.pt", recursive=True):
    d = os.path.dirname(f); name = os.path.basename(d)
    if not os.path.exists(f"{OUT}/{name}") and os.path.exists(f"{d}/test_prob.pt"):
        shutil.copytree(d, f"{OUT}/{name}", ignore=shutil.ignore_patterns("*.zip", "train_logits.pt", "model.pt"))
        print("main run copied:", name)
def ens(name, runs):
    runs = [f"{OUT}/{r}" for r in runs if os.path.exists(f"{OUT}/{r}/test_prob.pt")]
    if len(runs) >= 2:
        r = subprocess.run(["python", "p5_lsi.py", "ensemble", "--data", DATA, "--out", f"{OUT}/{name}", *runs],
                           capture_output=True, text=True)
        print(name, (r.stdout + r.stderr).strip().splitlines()[-1])
dvl, _ = P5.dev_labels(DATA)
cands = {os.path.basename(os.path.dirname(f)): torch.load(f) for f in glob.glob(f"{OUT}/*/dev_prob.pt")
         if not os.path.basename(os.path.dirname(f)).startswith(("ens_", "post"))}
def dev_of(names):
    lp = {k: sum(cands[n][k].float() for n in names) / len(names) for k in cands[names[0]]}
    return P5.quick_dev_f1(dvl, lp)
ranked = sorted(cands, key=lambda n: -dev_of([n]))
print("single-run dev (global thr):", {n: round(dev_of([n]), 4) for n in ranked})
for a, b in [("unilm_full", "unilm_full_cue"), ("unilm_lorma", "unilm_lorma_cue")]:
    if a in cands and b in cands:
        print(f"ABLATION {b} vs {a}: {dev_of([b]):.4f} vs {dev_of([a]):.4f}  (pair ensemble {dev_of([a, b]):.4f})")
sel = ranked[:1]
while len(sel) < len(ranked):
    best = max((dev_of(sel + [n]), n) for n in ranked if n not in sel)
    if best[0] < dev_of(sel) - 0.002:
        break
    sel.append(best[1])
print("greedy selection:", sel, round(dev_of(sel), 4))
ens("ens_greedy2", sel)
ens("ens_cue_pairs", [n for n in ["unilm_full", "unilm_full_cue", "unilm_lorma", "unilm_lorma_cue"] if n in cands])
ens("ens_all2", [n for n in ranked if not n.startswith("encdec") and n != "unilm_head"])
best_ens = f"{OUT}/ens_greedy2" if os.path.exists(f"{OUT}/ens_greedy2/test_prob.pt") else f"{OUT}/{ranked[0]}"
r = subprocess.run(["python", "p5_post.py", "--data", DATA, "--out", f"{OUT}/post", "--variants", "base,cooc,resid", best_ens],
                   capture_output=True, text=True)
print("post-processing on", os.path.basename(best_ens)); print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if l.split(" ")[0] in ("base", "cooc", "resid") or "Error" in l))

In [ ]:
import json, pandas as pd
rows = []
for d in sorted(glob.glob(f"{OUT}/*/run.json")):
    r = json.load(open(d)); name = os.path.basename(os.path.dirname(d))
    rows.append({"run": name, "arch": r.get("arch"), "method": r.get("peft", {}).get("method"),
                 "trainable_M": round(r.get("trainable", 0) / 1e6, 2),
                 "dev_honest": r.get("dev_f1_honest"), "dev_tuned": r.get("dev_f1_tuned"),
                 "dev@0.5": r.get("dev_f1_at_0.5"), "rule": r.get("rule"), "micro": r.get("dev_micro_tuned"),
                 "labels/doc": r.get("dev_labels_per_doc_pred"),
                 "epoch_min": round(((r.get("history") or [{}])[-1].get("sec") or 0) / 60, 1),
                 "zip": os.path.exists(f"{OUT}/{name}/submission.zip")})
df = pd.DataFrame(rows).sort_values("dev_honest", ascending=False, na_position="last")
print(df.to_string(index=False))
df.to_csv(f"{OUT}/summary.csv", index=False)
import zipfile
with zipfile.ZipFile(f"{WORK}/p5r2_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/**/*", recursive=True):
        if os.path.isfile(f) and not f.endswith(".pt"):
            z.write(f, os.path.relpath(f, WORK))
    for f in glob.glob(f"{OUT}/*/*_prob.pt"):
        z.write(f, os.path.relpath(f, WORK))
print("p5r2_results.zip", os.path.getsize(f"{WORK}/p5r2_results.zip"), "bytes")